# Fraud Threshold Optimisation under PSR/FCA Regulation — v7
Supersedes v6 - includes everything v6 has (PaySim as a
third robustness dataset, the wider IEEE-CIS feature pass) plus ONE new
addition: a logistic-stacked ensemble over the three proposal models
(Cell 22), reported as a supplementary, clearly-labelled exploration
alongside - not instead of - the three-model comparison the proposal
specifies. This is the one remaining lever with genuine headroom on
AUC-PR: calibration technique cannot move it (a ranking-only metric,
invariant to the monotonic isotonic transform already applied),
See `README_v7.md` for run instructions.

In [ ]:
# =============================================================================
# CELL 1 - ENVIRONMENT
# =============================================================================
!pip install -q xgboost imbalanced-learn shap

import importlib
import sys

for _pkg in ["numpy", "pandas", "sklearn", "xgboost", "imblearn", "shap", "matplotlib", "scipy"]:
    _m = importlib.import_module(_pkg)
    print(f"{_pkg:<12} {getattr(_m, '__version__', 'n/a')}")
print(f"{'python':<12} {sys.version.split()[0]}")

!free -g | head -2

numpy        2.1.3
pandas       2.2.3
sklearn      1.6.1
xgboost      3.4.1
imblearn     0.14.2
shap         0.52.0
matplotlib   3.10.0
scipy        1.16.3
python       3.13.15
               total        used        free      shared  buff/cache   available
Mem:              12           1           7           0           3          11


In [ ]:
# =============================================================================
# CELL 2 - IMPORTS AND CONFIGURATION
# =============================================================================

from __future__ import annotations

import gc
import json
import os
import time
import warnings
import zipfile
from collections import deque
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "font.size": 10,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

STUDENT_ID = "5753132"


DATA_DIR = Path(os.environ.get("DISSERTATION_DATA_DIR", "/content/data"))
OUT_DIR = Path(os.environ.get("DISSERTATION_OUT_DIR", "/content/outputs"))
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
for _d in (DATA_DIR, OUT_DIR, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    # ---- data ---------------------------------------------------------
    dataset: str = "ieee"          # "ieee" | "ulb" | "paysim" | "synthetic"

    sample_rows: int | None = 200_000
    usd_to_gbp: float = 0.79
    eur_to_gbp: float = 0.85
    app_uplift: float = 1.0

    # ---- splitting ------------------------------------------------------
    split: str = "stratified"       # "stratified" | "temporal"
    train_frac: float = 0.70
    val_frac: float = 0.15
    random_state: int = 42

    # ---- preprocessing ----------------------------------------------------
    max_missing_frac: float = 0.90
    min_unique: int = 2
    freq_encode_cardinality: int = 2

    # "smote" | "borderline_smote" | "smote_tomek" | "class_weight" | "none".
    imbalance_method: str = "smote"
    smote_ratio: float = 0.10
    smote_k_neighbors: int = 5
    prior_correction: bool = True

    # ---- calibration ------------------------------------------------------
    recalibration: str = "isotonic"  # "isotonic" | "platt" | "none"

    # ---- models -------------------------------------------------------
    models: tuple = ("logistic", "random_forest", "xgboost")

    # ---- costs ------------------------------------------------------------
    regime: str = "uk_regulatory"
    threshold_grid: int = 1001


    precision_target: float = 0.90
    recall_target: float = 0.90

    # ---- model hyperparameters (overwritten by Cell 10 if tune_hyperparams) --
    lr_C: float = 1.0
    lr_max_iter: int = 2000
    rf_n_estimators: int = 300
    rf_max_depth: int | None = 18
    rf_min_samples_leaf: int = 5
    xgb_n_estimators: int = 2000
    xgb_learning_rate: float = 0.05
    xgb_max_depth: int = 6
    xgb_subsample: float = 0.8
    xgb_colsample_bytree: float = 0.8
    xgb_early_stopping: int = 50
    xgb_search_n_estimators: int = 400  # budget used only during Cell 10 search
    xgb_device: str = "cpu"             # "cpu" | "cuda" - set "cuda" on a GPU runtime
    n_jobs: int = -1


    tune_hyperparams: bool = False
    hp_search_iter: int = 10

    run_shap: bool = False
    shap_max_rows: int = 2000


    bootstrap_n: int = 500
    bootstrap_grid: int = 401


    seed_stability_sample: int = 150_000


CFG = Config()

PSR_CAP_GBP = 85_000.0              # PSR PS23/3 s.3 - mandatory reimbursement cap
PSR_LIABILITY_SHARE = 0.50          # PS23/3 s.3.2 - 50/50 sending/receiving PSP split
PSR_EXCESS_GBP = 0.0                # PS23/3 permits, does not require, a GBP 100 excess
CLAIM_ADMIN_GBP = 45.0              # UK Finance complaint-handling benchmark, per claim

REGIMES = {
    "literature": {
        "fp_cost": 10.0, "tp_cost": 5.0, "tn_cost": 0.0,
        "fn_mode": "amount", "fn_share": 1.0, "fn_cap": None,
        "fn_flat": None, "fn_admin": 0.0, "fn_excess": 0.0,
    },
    "uk_regulatory": {
        "fp_cost": 35.0, "tp_cost": 8.0, "tn_cost": 0.0,
        "fn_mode": "capped_amount", "fn_share": PSR_LIABILITY_SHARE,
        "fn_cap": PSR_CAP_GBP, "fn_flat": None,
        "fn_admin": CLAIM_ADMIN_GBP, "fn_excess": PSR_EXCESS_GBP,
    },
    "uk_statutory_cap": {
        "fp_cost": 35.0, "tp_cost": 8.0, "tn_cost": 0.0,
        "fn_mode": "flat", "fn_share": 1.0, "fn_cap": None,
        "fn_flat": PSR_CAP_GBP, "fn_admin": 0.0, "fn_excess": 0.0,
    },
}


FP_COST_SWEEP = [10.0, 25.0, 35.0, 50.0, 75.0, 100.0, 120.0]
CAP_SWEEP = [1_000.0, 5_000.0, 15_000.0, 30_000.0, 85_000.0, 415_000.0]
APP_UPLIFT_SWEEP = [1.0, 5.0, 10.0, 20.0]
LIABILITY_SHARE_SWEEP = [0.0, 0.25, 0.50, 0.75, 1.0]
CLAIM_ADMIN_SWEEP = [0.0, 25.0, 45.0, 75.0, 120.0]


SAMPLE_SIZE_ABLATION = [50_000, 100_000, 200_000, None]

# Hyperparameter search runs on this subsample, never on the full dataset
TUNING_SAMPLE_ROWS = 150_000

_LOG = True


def log(*args, **kwargs):
    if _LOG:
        print(*args, **kwargs)


RESULTS: list[dict] = []
ARTEFACTS: dict = {}
TUNING_TRACES: dict[str, pd.DataFrame] = {}
SHAP_RESULTS: dict[str, dict] = {}
TUNED_CONFIGS: dict[str, Config] = {}   # run_id -> cfg with searched hyperparameters applied

np.random.seed(CFG.random_state)
print(json.dumps(asdict(CFG), indent=2, default=str))

{
  "dataset": "ieee",
  "sample_rows": 200000,
  "usd_to_gbp": 0.79,
  "eur_to_gbp": 0.85,
  "app_uplift": 1.0,
  "split": "stratified",
  "train_frac": 0.7,
  "val_frac": 0.15,
  "random_state": 42,
  "max_missing_frac": 0.9,
  "min_unique": 2,
  "freq_encode_cardinality": 2,
  "imbalance_method": "smote",
  "smote_ratio": 0.1,
  "smote_k_neighbors": 5,
  "prior_correction": true,
  "recalibration": "isotonic",
  "models": [
    "logistic",
    "random_forest",
    "xgboost"
  ],
  "regime": "uk_regulatory",
  "threshold_grid": 1001,
  "precision_target": 0.9,
  "recall_target": 0.9,
  "lr_C": 1.0,
  "lr_max_iter": 2000,
  "rf_n_estimators": 300,
  "rf_max_depth": 18,
  "rf_min_samples_leaf": 5,
  "xgb_n_estimators": 2000,
  "xgb_learning_rate": 0.05,
  "xgb_max_depth": 6,
  "xgb_subsample": 0.8,
  "xgb_colsample_bytree": 0.8,
  "xgb_early_stopping": 50,
  "xgb_search_n_estimators": 400,
  "xgb_device": "cpu",
  "n_jobs": -1,
  "tune_hyperparams": false,
  "hp_search_iter": 10,
  "ru

In [ ]:
# =============================================================================
# CELL 3 - DATA ACQUISITION
# =============================================================================

AUTH = "token"        # "token" | "oauth" | "legacy_json"
SOURCE = "kaggle"     # "kaggle" | "drive" | "upload" | "none"
FETCH_PAYSIM = False  # set True to also pull the third robustness dataset (~470MB)

!pip install -q --upgrade kaggle

if SOURCE == "kaggle":
    KAGGLE_DIR = Path.home() / ".kaggle"
    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

    if AUTH == "token":
        from getpass import getpass
        _tok = getpass("Paste Kaggle API token (KGAT_...): ").strip()
        _tf = KAGGLE_DIR / "access_token"
        _tf.write_text(_tok)
        _tf.chmod(0o600)
        os.environ["KAGGLE_API_TOKEN"] = _tok
        del _tok
    elif AUTH == "oauth":
        !kaggle auth login
    elif AUTH == "legacy_json":
        from google.colab import files
        if not (KAGGLE_DIR / "kaggle.json").exists():
            print("Upload kaggle.json")
            files.upload()
            os.replace("kaggle.json", KAGGLE_DIR / "kaggle.json")
            (KAGGLE_DIR / "kaggle.json").chmod(0o600)

    print("\n--- auth check ---")
    !kaggle competitions files ieee-fraud-detection 2>&1 | head -8

if SOURCE == "kaggle":
    !kaggle competitions download -c ieee-fraud-detection -f train_transaction.csv -p {DATA_DIR}
    !kaggle competitions download -c ieee-fraud-detection -f train_identity.csv -p {DATA_DIR}
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p {DATA_DIR}
    if FETCH_PAYSIM:
        !kaggle datasets download -d ealaxi/paysim1 -p {DATA_DIR}

elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/dissertation/data")
    for _f in DRIVE_DATA.glob("*"):
        if _f.suffix in {".csv", ".zip"}:
            !cp "{_f}" {DATA_DIR}/

elif SOURCE == "upload":
    from google.colab import files
    files.upload()
    for _f in Path(".").glob("*.zip"):
        _f.replace(DATA_DIR / _f.name)

for _z in DATA_DIR.glob("*.zip"):
    with zipfile.ZipFile(_z) as zf:
        zf.extractall(DATA_DIR)
    _z.unlink()

for _f in sorted(DATA_DIR.glob("*.csv")):
    print(f"{_f.name:<28} {_f.stat().st_size / 1e6:8.1f} MB")

_missing = [f for f in ("train_transaction.csv", "creditcard.csv") if not (DATA_DIR / f).exists()]
print("MISSING:", _missing if _missing else "nothing - ready to run")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.0 MB/s eta 0:00:00
Paste Kaggle API token (KGAT_...): ··········

--- auth check ---
name                         size  creationDate                
---------------------  ----------  --------------------------  
sample_submission.csv     6080314  2019-12-11 23:08:51.100000  
test_identity.csv        25797161  2019-12-11 23:08:51.619000  
test_transaction.csv    613194934  2019-12-11 23:09:18.466000  
train_identity.csv       26529680  2019-12-11 23:09:00.658000  
train_transaction.csv   683351067  2019-12-11 23:09:14.423000  
100% 58.3M/58.3M [00:00<00:00, 192MB/s]

100% 3.26M/3.26M [00:00<00:00, 118MB/s]

Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 147MB/s]

creditcard.csv                  150.8 MB
train_identity.csv               26.5 MB
train_transaction.csv           683.4 MB
MISSING: nothing - ready to run


In [ ]:
# =============================================================================
# CELL 4 - DATA LOADING
# =============================================================================


def _downcast(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = df[c].astype("float32")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


def load_ieee(cfg: Config) -> tuple:
    """IEEE-CIS Fraud Detection. 590,540 rows, ~3.5% fraud, USD."""
    tx = pd.read_csv(DATA_DIR / "train_transaction.csv")
    tx = _downcast(tx)
    idf = DATA_DIR / "train_identity.csv"
    if idf.exists():
        ident = _downcast(pd.read_csv(idf))
        tx = tx.merge(ident, on="TransactionID", how="left")
        del ident
        gc.collect()

    y = tx.pop("isFraud").astype("int8").to_numpy()
    order = tx["TransactionDT"].astype("float64").to_numpy()
    amount = tx["TransactionAmt"].astype("float32").to_numpy() * cfg.usd_to_gbp * cfg.app_uplift
    X = tx.drop(columns=["TransactionID"])
    return X, y, amount.astype("float32"), order


def load_ulb(cfg: Config) -> tuple:
    """ULB Credit Card Fraud. 284,807 rows, 0.172% fraud, EUR, PCA features."""
    df = _downcast(pd.read_csv(DATA_DIR / "creditcard.csv"))
    y = df.pop("Class").astype("int8").to_numpy()
    order = df["Time"].astype("float64").to_numpy()
    amount = df["Amount"].astype("float32").to_numpy() * cfg.eur_to_gbp * cfg.app_uplift
    return df, y, amount.astype("float32"), order


def load_paysim(cfg: Config) -> tuple:
    """PaySim mobile-money fraud (Lopez-Rojas et al., 2016), Kaggle
    ealaxi/paysim1. Third independent robustness dataset (NEW in v6) - the
    closest public proxy to push-payment (APP) fraud, since it models
    person-to-person mobile transfers rather than card-present/CNP card
    fraud like IEEE-CIS and ULB. Cannot be merged into IEEE-CIS's training
    set (incompatible schema) - fit and reported exactly like ULB."""
    fname = next(DATA_DIR.glob("PS_2017*.csv"), None) or DATA_DIR / "paysim.csv"
    df = _downcast(pd.read_csv(fname))
    y = df.pop("isFraud").astype("int8").to_numpy()
    df = df.drop(columns=[c for c in ("isFlaggedFraud", "nameOrig", "nameDest") if c in df.columns])
    order = df["step"].astype("float64").to_numpy()
    amount = df["amount"].astype("float32").to_numpy() * cfg.app_uplift
    X = df.drop(columns=["amount"])
    return X, y, amount.astype("float32"), order


def load_synthetic(cfg: Config, n: int = 40_000) -> tuple:
    """Imbalanced stand-in used to smoke-test the pipeline without any download."""
    rng = np.random.default_rng(cfg.random_state)
    y = (rng.random(n) < 0.03).astype("int8")
    sig = rng.normal(y * 0.55, 1.0, size=(6, n)).T
    noise = rng.normal(0, 1, size=(n, 14))
    X = pd.DataFrame(np.hstack([sig, noise]).astype("float32"), columns=[f"f{i}" for i in range(20)])
    X["cat_a"] = rng.choice(list("abcdef"), n)
    X.loc[rng.random(n) < 0.15, "f3"] = np.nan
    amount = (rng.lognormal(3.6, 1.1, n) * (1 + 2.0 * y)).astype("float32")
    order = np.sort(rng.random(n) * 1e6)
    return X, y, amount * cfg.app_uplift, order


def load_dataset(cfg: Config) -> tuple:
    loader = {"ieee": load_ieee, "ulb": load_ulb, "paysim": load_paysim, "synthetic": load_synthetic}[cfg.dataset]
    X, y, amount, order = loader(cfg)

    if cfg.sample_rows is not None and cfg.sample_rows < len(y):
        idx, _ = train_test_split(
            np.arange(len(y)), train_size=cfg.sample_rows, stratify=y, random_state=cfg.random_state
        )
        idx = np.sort(idx)
        X, y, amount, order = X.iloc[idx].reset_index(drop=True), y[idx], amount[idx], order[idx]

    gc.collect()
    log(
        f"{cfg.dataset}: {len(y):,} rows x {X.shape[1]} cols | "
        f"fraud {y.mean():.4%} ({int(y.sum()):,}) | "
        f"amount GBP mean {amount.mean():,.2f} median {np.median(amount):,.2f} max {amount.max():,.2f}"
    )
    return X, y, amount, order

In [ ]:
# =============================================================================
# CELL 5 - FEATURE ENGINEERING
# =============================================================================


def engineer_features(X: pd.DataFrame, amount: np.ndarray, order: np.ndarray, cfg: Config) -> pd.DataFrame:
    X = X.copy()

    X["fe_log_amount"] = np.log1p(np.maximum(amount, 0)).astype("float32")
    X["fe_amount_decimal"] = (amount - np.floor(amount)).astype("float32")
    X["fe_amount_is_round"] = (X["fe_amount_decimal"] < 1e-6).astype("int8")
    X["fe_n_missing"] = X.isna().sum(axis=1).astype("int32")

    hour = (order / 3600.0) % 24.0
    X["fe_hour"] = hour.astype("float32")
    X["fe_hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
    X["fe_hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")
    X["fe_dayofweek"] = ((order / 86400.0) % 7).astype("float32")

    if cfg.dataset == "ieee":
        for col in ("P_emaildomain", "R_emaildomain"):
            if col in X.columns:
                X[f"fe_{col}_root"] = X[col].astype("string").str.split(".").str[0]
        if {"P_emaildomain", "R_emaildomain"} <= set(X.columns):
            X["fe_email_match"] = (
                X["P_emaildomain"].astype("string").eq(X["R_emaildomain"].astype("string"))
                .astype("Int8").fillna(-1).astype("int8")
            )
        if "DeviceInfo" in X.columns:
            X["fe_device_root"] = X["DeviceInfo"].astype("string").str.split("/").str[0]

        # Velocity features. Leak-safe: uses only transaction time
        # and card identity, never the label, and only each row's OWN prior
        if "card1" in X.columns:
            ord_idx = np.argsort(order, kind="mergesort")
            card_sorted = X["card1"].to_numpy()[ord_idx]
            time_sorted = order[ord_idx]
            count_so_far = np.zeros(len(order), dtype="float32")
            secs_since_prev = np.full(len(order), -1.0, dtype="float32")
            count_last_1h = np.zeros(len(order), dtype="float32")
            last_count: dict = {}
            last_time: dict = {}

            windows: dict = {}
            WINDOW_SECONDS = 3600.0
            for i in range(len(ord_idx)):
                c, t = card_sorted[i], time_sorted[i]
                if c in last_count:
                    count_so_far[i] = last_count[c]
                    secs_since_prev[i] = t - last_time[c]
                dq = windows.setdefault(c, deque())
                while dq and t - dq[0] > WINDOW_SECONDS:
                    dq.popleft()
                count_last_1h[i] = len(dq)
                dq.append(t)
                last_count[c] = last_count.get(c, 0) + 1.0
                last_time[c] = t
            inv = np.empty_like(ord_idx)
            inv[ord_idx] = np.arange(len(ord_idx))
            X["fe_card1_count_so_far"] = count_so_far[inv]
            X["fe_card1_secs_since_prev"] = secs_since_prev[inv]
            X["fe_card1_count_last_1h"] = count_last_1h[inv]

    if cfg.dataset == "paysim":
        for a, b in (("oldbalanceOrg", "newbalanceOrig"), ("oldbalanceDest", "newbalanceDest")):
            if a in X.columns and b in X.columns:
                X[f"fe_{a}_delta"] = (X[a] - X[b]).astype("float32")

    X = X.drop(columns=[c for c in ("TransactionDT", "Time", "step") if c in X.columns])
    return X

In [ ]:
# =============================================================================
# CELL 6 - SPLITTING
# 70/15/15 train/val/test,
# =============================================================================


def make_splits(y: np.ndarray, order: np.ndarray, cfg: Config) -> dict[str, np.ndarray]:
    n = len(y)
    idx = np.arange(n)

    if cfg.split == "temporal":
        srt = idx[np.argsort(order, kind="mergesort")]
        i_tr = int(cfg.train_frac * n)
        i_va = int((cfg.train_frac + cfg.val_frac) * n)
        tr, va, te = srt[:i_tr], srt[i_tr:i_va], srt[i_va:]
    elif cfg.split == "stratified":
        tr, rest = train_test_split(idx, train_size=cfg.train_frac, stratify=y, random_state=cfg.random_state)
        va_share = cfg.val_frac / (1.0 - cfg.train_frac)
        va, te = train_test_split(rest, train_size=va_share, stratify=y[rest], random_state=cfg.random_state)
    else:
        raise ValueError(f"unknown split: {cfg.split}")

    va_cal, va_thr = train_test_split(va, train_size=0.5, stratify=y[va], random_state=cfg.random_state)

    sp = {"train": np.sort(tr), "val_cal": np.sort(va_cal), "val_thr": np.sort(va_thr), "test": np.sort(te)}
    for k, v in sp.items():
        log(f"{k:<9} n={len(v):>8,}  fraud={y[v].mean():.4%}  positives={int(y[v].sum()):>6,}")
    return sp

In [ ]:
# =============================================================================
# CELL 7 - LEAK-SAFE PREPROCESSING
# All statistics (missingness, frequencies, medians, scaler) are fit on
# TRAIN ROWS ONLY, then applied unchanged to val/test.
# =============================================================================


# Train-only aggregate-encoding key groups (mean/std/count of TransactionAmt).
AGG_KEY_GROUPS = (
    ("card1",), ("addr1",), ("card1", "addr1"),
    ("card2",), ("P_emaildomain",), ("card1", "P_emaildomain"),
)


class FeaturePreparer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.keep_: list[str] = []
        self.cat_cols_: list[str] = []
        self.num_cols_: list[str] = []
        self.freq_maps_: dict[str, dict] = {}
        self.medians_: pd.Series | None = None
        self.scaler_: StandardScaler | None = None
        self.agg_stats_: dict[tuple, dict] = {}   # keys -> {"mean":{}, "std":{}, "count":{}}
        self.global_amt_mean_ = 0.0
        self.global_amt_std_ = 0.0

    def fit(self, X: pd.DataFrame) -> "FeaturePreparer":
        miss = X.isna().mean()
        nuniq = X.nunique(dropna=True)
        drop = set(miss[miss > self.cfg.max_missing_frac].index) | set(nuniq[nuniq < self.cfg.min_unique].index)
        self.keep_ = [c for c in X.columns if c not in drop]
        Xk = X[self.keep_]

        self.cat_cols_ = [c for c in Xk.columns if not pd.api.types.is_numeric_dtype(Xk[c])]
        self.num_cols_ = [c for c in Xk.columns if c not in self.cat_cols_]

        for c in self.cat_cols_:
            vc = Xk[c].astype("string").value_counts(normalize=True)
            self.freq_maps_[c] = vc.to_dict()

        if "TransactionAmt" in Xk.columns:
            self.global_amt_mean_ = float(Xk["TransactionAmt"].mean())
            self.global_amt_std_ = float(Xk["TransactionAmt"].std())
            for keys in AGG_KEY_GROUPS:
                if all(k in Xk.columns for k in keys):
                    grp = Xk.groupby(list(keys))["TransactionAmt"]
                    self.agg_stats_[keys] = {
                        "mean": grp.mean().to_dict(),
                        "std": grp.std().fillna(0.0).to_dict(),
                        "count": grp.size().astype("float32").to_dict(),
                    }

        self.medians_ = Xk[self.num_cols_].median()
        self.scaler_ = StandardScaler().fit(self._core(Xk))
        log(
            f"prepared: kept {len(self.keep_)}/{X.shape[1]} cols "
            f"({len(self.num_cols_)} numeric, {len(self.cat_cols_)} categorical), dropped {len(drop)}, "
            f"aggregate groups on: {list(self.agg_stats_.keys())}"
        )
        return self

    def _agg_key_series(self, Xk: pd.DataFrame, keys: tuple) -> pd.Series:
        if len(keys) == 1:
            return Xk[keys[0]]
        return pd.Series(list(zip(*(Xk[k] for k in keys))), index=Xk.index)

    def _agg_name(self, keys: tuple, stat: str) -> str:
        return f"fe_{'_'.join(keys)}_amt_{stat}"

    def _core(self, Xk: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=Xk.index)
        for c in self.num_cols_:
            out[c] = Xk[c].astype("float32")
        for c in self.cat_cols_:
            out[c] = Xk[c].astype("string").map(self.freq_maps_[c]).fillna(0.0).astype("float32")
        out[self.num_cols_] = out[self.num_cols_].fillna(self.medians_)
        for keys, stats in self.agg_stats_.items():
            key_series = self._agg_key_series(Xk, keys)
            out[self._agg_name(keys, "mean")] = key_series.map(stats["mean"]).fillna(self.global_amt_mean_).astype("float32")
            out[self._agg_name(keys, "std")] = key_series.map(stats["std"]).fillna(self.global_amt_std_).astype("float32")
            out[self._agg_name(keys, "count")] = key_series.map(stats["count"]).fillna(0.0).astype("float32")
        return out.fillna(0.0)

    def transform(self, X: pd.DataFrame, scale: bool = False) -> np.ndarray:
        core = self._core(X[self.keep_])
        arr = self.scaler_.transform(core) if scale else core.to_numpy()
        return np.ascontiguousarray(arr, dtype="float32")

    @property
    def feature_names_(self) -> list[str]:
        extra = [self._agg_name(keys, stat) for keys in self.agg_stats_ for stat in ("mean", "std", "count")]
        return self.num_cols_ + self.cat_cols_ + extra

In [ ]:
# =============================================================================
# CELL 8 - CLASS IMBALANCE
# SMOTE on TRAINING ROWS ONLY.
# =============================================================================


def resample_train(X_tr: np.ndarray, y_tr: np.ndarray, cfg: Config) -> tuple:
    pi_orig = float(y_tr.mean())

    if cfg.imbalance_method in ("none", "class_weight"):
        # No resampling under either method - "none" fits on the raw 3.5%
        # prior as-is, "class_weight" reweights the loss instead of touching
        # the data (see build_model). pi_res == pi_orig makes prior_correct()
        # a no-op automatically, which is the correct behaviour here: there
        # is no resampled distribution to correct back from.
        log(f"imbalance_method={cfg.imbalance_method}: no resampling | prior {pi_orig:.4%}")
        return X_tr, y_tr, pi_orig, pi_orig

    current = y_tr.sum() / max((y_tr == 0).sum(), 1)
    if current >= cfg.smote_ratio:
        log(f"{cfg.imbalance_method}: skipped, existing ratio {current:.4f} >= target {cfg.smote_ratio}")
        return X_tr, y_tr, pi_orig, pi_orig

    if cfg.imbalance_method == "smote":
        sampler = SMOTE(sampling_strategy=cfg.smote_ratio, k_neighbors=cfg.smote_k_neighbors, random_state=cfg.random_state)
    elif cfg.imbalance_method == "borderline_smote":
        # Only oversamples minority points near the decision boundary
        # ("in danger" - surrounded by more majority than minority
        # neighbours), rather than uniformly across the whole minority
        # class. Same cost profile as plain SMOTE.
        sampler = BorderlineSMOTE(sampling_strategy=cfg.smote_ratio, k_neighbors=cfg.smote_k_neighbors, random_state=cfg.random_state)
    elif cfg.imbalance_method == "smote_tomek":
        # SMOTE oversampling, then Tomek-link removal deletes overlapping
        # majority/minority pairs. Tomek linking is an O(n log n) nearest
        # -neighbour search over the WHOLE resampled set in ~440 dimensions -
        # cheap at the tuning-subsample scale this is compared at (Cell 19),
        # but do not point this at the full 590k-row primary_cfg without
        # timing it first. See README_v7.md.
        _smote = SMOTE(sampling_strategy=cfg.smote_ratio, k_neighbors=cfg.smote_k_neighbors, random_state=cfg.random_state)
        sampler = SMOTETomek(sampling_strategy=cfg.smote_ratio, smote=_smote, random_state=cfg.random_state)
    else:
        raise ValueError(f"unknown imbalance_method: {cfg.imbalance_method}")

    X_res, y_res = sampler.fit_resample(X_tr, y_tr)
    pi_res = float(y_res.mean())
    log(f"{cfg.imbalance_method}: {len(y_tr):,} -> {len(y_res):,} rows | prior {pi_orig:.4%} -> {pi_res:.4%}")
    gc.collect()
    return X_res.astype("float32"), y_res, pi_orig, pi_res

In [ ]:
# =============================================================================
# CELL 9 - MODELS
# =============================================================================

MODEL_LABELS = {"logistic": "Logistic Regression", "random_forest": "Random Forest", "xgboost": "XGBoost"}
NEEDS_SCALING = {"logistic": True, "random_forest": True, "xgboost": True}


def build_model(name: str, cfg: Config, y_res: np.ndarray | None = None):
    # class_weight/scale_pos_weight only apply when imbalance_method=
    # "class_weight" (no resampling happened, so y_res is still the raw
    # ~3.5% distribution). Any other method already balanced the data itself
    # - stacking a class weight on top of that would double-count it.
    use_cw = cfg.imbalance_method == "class_weight"

    if name == "logistic":
        return LogisticRegression(
            C=cfg.lr_C, max_iter=cfg.lr_max_iter, solver="lbfgs", n_jobs=cfg.n_jobs,
            random_state=cfg.random_state, class_weight="balanced" if use_cw else None,
        )

    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=cfg.rf_n_estimators, max_depth=cfg.rf_max_depth,
            min_samples_leaf=cfg.rf_min_samples_leaf, n_jobs=cfg.n_jobs, random_state=cfg.random_state,
            class_weight="balanced" if use_cw else None,
        )

    if name == "xgboost":
        spw = 1.0
        if use_cw and y_res is not None:
            pos, neg = float((y_res == 1).sum()), float((y_res == 0).sum())
            spw = neg / max(pos, 1.0)
        return XGBClassifier(
            n_estimators=cfg.xgb_n_estimators, learning_rate=cfg.xgb_learning_rate, max_depth=cfg.xgb_max_depth,
            subsample=cfg.xgb_subsample, colsample_bytree=cfg.xgb_colsample_bytree,
            objective="binary:logistic", eval_metric="aucpr", early_stopping_rounds=cfg.xgb_early_stopping,
            tree_method="hist", device=cfg.xgb_device, n_jobs=cfg.n_jobs, random_state=cfg.random_state,
            scale_pos_weight=spw,
        )

    raise ValueError(f"unknown model: {name}")


def fit_model(name: str, cfg: Config, X_res, y_res, X_cal, y_cal):
    model = build_model(name, cfg, y_res)
    t0 = time.time()
    if name == "xgboost":
        model.fit(X_res, y_res, eval_set=[(X_cal, y_cal)], verbose=False)
        log(f"{name}: fitted in {time.time() - t0:.1f}s | best_iteration={getattr(model, 'best_iteration', None)}")
    else:
        model.fit(X_res, y_res)
        log(f"{name}: fitted in {time.time() - t0:.1f}s")
    return model

In [ ]:
# =============================================================================
# CELL 10 - HYPERPARAMETER SEARCH
# =============================================================================

# RF has no early stopping and no GPU path. max_depth=None + min_samples_leaf=1
# grows trees to full purity - on ~500k rows that can take an hour on its own
# for ONE fit - and stay excluded here for that reason.
RF_SEARCH_SPACE = {
    "n_estimators": [150, 200, 300, 400],
    "max_depth": [10, 14, 18, 22],
    "min_samples_leaf": [3, 5, 8],
}
XGB_SEARCH_SPACE = {
    "max_depth": [4, 5, 6, 8],
    "learning_rate": [0.02, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}
# C=100 (weak regularisation) on ~430 correlated columns is a known slow
# -convergence trap for lbfgs - excluded after a live run stalled on it.
LR_SEARCH_SPACE = {"C": [0.01, 0.1, 1.0, 10.0]}


def _sample_params(space: dict, rng: np.random.Generator) -> dict:
    return {k: v[int(rng.integers(len(v)))] for k, v in space.items()}


def tune_random_forest(cfg: Config, X_res, y_res, X_cal, y_cal) -> tuple[dict, pd.DataFrame]:
    use_cw = cfg.imbalance_method == "class_weight"
    rng = np.random.default_rng(cfg.random_state)
    best_score, best_params, trials = -np.inf, dict(RF_SEARCH_SPACE.copy()), []
    for i in range(cfg.hp_search_iter):
        params = _sample_params(RF_SEARCH_SPACE, rng)
        t0 = time.time()
        m = RandomForestClassifier(**params, n_jobs=cfg.n_jobs, random_state=cfg.random_state,
                                    class_weight="balanced" if use_cw else None)
        m.fit(X_res, y_res)
        score = average_precision_score(y_cal, m.predict_proba(X_cal)[:, 1])
        log(f"  rf [{i + 1}/{cfg.hp_search_iter}] {params} -> AUC-PR={score:.4f} ({time.time() - t0:.1f}s)")
        trials.append({**params, "auc_pr_val_cal": score})
        if score > best_score:
            best_score, best_params = score, params
        del m
        gc.collect()
    log(f"random_forest tuned: best AUC-PR(val_cal)={best_score:.4f} params={best_params}")
    return best_params, pd.DataFrame(trials)


def tune_xgboost(cfg: Config, X_res, y_res, X_cal, y_cal) -> tuple[dict, pd.DataFrame]:
    spw = 1.0
    if cfg.imbalance_method == "class_weight":
        pos, neg = float((y_res == 1).sum()), float((y_res == 0).sum())
        spw = neg / max(pos, 1.0)
    rng = np.random.default_rng(cfg.random_state)
    best_score, best_params, trials = -np.inf, {}, []
    for i in range(cfg.hp_search_iter):
        params = _sample_params(XGB_SEARCH_SPACE, rng)
        t0 = time.time()
        m = XGBClassifier(
            n_estimators=cfg.xgb_search_n_estimators, **params, objective="binary:logistic",
            eval_metric="aucpr", early_stopping_rounds=cfg.xgb_early_stopping,
            tree_method="hist", device=cfg.xgb_device, n_jobs=cfg.n_jobs, random_state=cfg.random_state,
            scale_pos_weight=spw,
        )
        m.fit(X_res, y_res, eval_set=[(X_cal, y_cal)], verbose=False)
        score = average_precision_score(y_cal, m.predict_proba(X_cal)[:, 1])
        log(f"  xgb [{i + 1}/{cfg.hp_search_iter}] {params} -> AUC-PR={score:.4f} ({time.time() - t0:.1f}s)")
        trials.append({**params, "auc_pr_val_cal": score})
        if score > best_score:
            best_score, best_params = score, params
        del m
        gc.collect()
    log(f"xgboost tuned: best AUC-PR(val_cal)={best_score:.4f} params={best_params}")
    return best_params, pd.DataFrame(trials)


def tune_logistic(cfg: Config, X_res, y_res, X_cal, y_cal) -> tuple[dict, pd.DataFrame]:
    use_cw = cfg.imbalance_method == "class_weight"
    best_score, best_C, trials = -np.inf, cfg.lr_C, []
    for C in LR_SEARCH_SPACE["C"]:
        t0 = time.time()
        m = LogisticRegression(C=C, max_iter=cfg.lr_max_iter, solver="lbfgs", n_jobs=cfg.n_jobs,
                                random_state=cfg.random_state, class_weight="balanced" if use_cw else None)
        m.fit(X_res, y_res)
        score = average_precision_score(y_cal, m.predict_proba(X_cal)[:, 1])
        log(f"  lr C={C} -> AUC-PR={score:.4f} ({time.time() - t0:.1f}s)")
        trials.append({"C": C, "auc_pr_val_cal": score})
        if score > best_score:
            best_score, best_C = score, C
        del m
        gc.collect()
    log(f"logistic tuned: best AUC-PR(val_cal)={best_score:.4f} C={best_C}")
    return {"C": best_C}, pd.DataFrame(trials)


def tune_hyperparameters(cfg: Config, X_res, y_res, X_cal, y_cal) -> tuple[Config, dict]:
    if not cfg.tune_hyperparams:
        return cfg, {}
    traces, new_fields = {}, {}
    if "random_forest" in cfg.models:
        best, trace = tune_random_forest(cfg, X_res, y_res, X_cal, y_cal)
        new_fields.update({"rf_n_estimators": best["n_estimators"], "rf_max_depth": best["max_depth"], "rf_min_samples_leaf": best["min_samples_leaf"]})
        traces["random_forest"] = trace
    if "xgboost" in cfg.models:
        best, trace = tune_xgboost(cfg, X_res, y_res, X_cal, y_cal)
        new_fields.update({"xgb_max_depth": best["max_depth"], "xgb_learning_rate": best["learning_rate"], "xgb_subsample": best["subsample"], "xgb_colsample_bytree": best["colsample_bytree"]})
        traces["xgboost"] = trace
    if "logistic" in cfg.models:
        best, trace = tune_logistic(cfg, X_res, y_res, X_cal, y_cal)
        new_fields["lr_C"] = best["C"]
        traces["logistic"] = trace
    return replace(cfg, **new_fields), traces

In [ ]:
# =============================================================================
# CELL 11 - PROBABILITY CALIBRATION
# Prior correction (Elkan 2001; Dal Pozzolo et al., 2015) undoes the SMOTE
# base-rate shift; isotonic/Platt recalibration then corrects residual shape
# distortion. ProbabilityChain.finalise() is the only route to a probability
# used anywhere downstream, so threshold selection and test scoring cannot
# drift onto different scales.
# =============================================================================

EPS = 1e-9


def prior_correct(p: np.ndarray, pi_res: float, pi_orig: float) -> np.ndarray:
    if abs(pi_res - pi_orig) < 1e-12:
        return p
    p = np.clip(p.astype("float64"), EPS, 1 - EPS)
    odds = p / (1.0 - p)
    factor = (pi_orig / (1.0 - pi_orig)) * ((1.0 - pi_res) / pi_res)
    corrected = odds * factor
    return corrected / (1.0 + corrected)


class ProbabilityChain:
    def __init__(self, name, model, preparer: FeaturePreparer, cfg: Config, pi_orig: float, pi_res: float):
        self.name = name
        self.model = model
        self.preparer = preparer
        self.cfg = cfg
        self.pi_orig = pi_orig
        self.pi_res = pi_res
        self.recalibrator = None

    def raw_proba(self, X_df: pd.DataFrame) -> np.ndarray:
        M = self.preparer.transform(X_df, scale=NEEDS_SCALING[self.name])
        return self.model.predict_proba(M)[:, 1].astype("float64")

    def fit_recalibrator(self, p_raw_cal: np.ndarray, y_cal: np.ndarray) -> "ProbabilityChain":
        p1 = prior_correct(p_raw_cal, self.pi_res, self.pi_orig) if self.cfg.prior_correction else p_raw_cal
        if self.cfg.recalibration == "isotonic":
            self.recalibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(p1, y_cal)
        elif self.cfg.recalibration == "platt":
            z = np.log(np.clip(p1, EPS, 1 - EPS) / (1 - np.clip(p1, EPS, 1 - EPS)))
            self.recalibrator = LogisticRegression(C=1e6, max_iter=1000).fit(z.reshape(-1, 1), y_cal)
        elif self.cfg.recalibration != "none":
            raise ValueError(f"unknown recalibration: {self.cfg.recalibration}")
        return self

    def finalise(self, p_raw: np.ndarray) -> np.ndarray:
        p = prior_correct(p_raw, self.pi_res, self.pi_orig) if self.cfg.prior_correction else p_raw.astype("float64")
        if self.recalibrator is None:
            return p
        if self.cfg.recalibration == "isotonic":
            return self.recalibrator.predict(p)
        z = np.log(np.clip(p, EPS, 1 - EPS) / (1 - np.clip(p, EPS, 1 - EPS)))
        return self.recalibrator.predict_proba(z.reshape(-1, 1))[:, 1]

    def predict_proba(self, X_df: pd.DataFrame) -> np.ndarray:
        return self.finalise(self.raw_proba(X_df))


def expected_calibration_error(y: np.ndarray, p: np.ndarray, bins: int = 20) -> float:
    edges = np.linspace(0.0, 1.0, bins + 1)
    b = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    ece = 0.0
    for k in range(bins):
        m = b == k
        if m.any():
            ece += (m.sum() / len(p)) * abs(p[m].mean() - y[m].mean())
    return float(ece)

In [ ]:
# =============================================================================
# CELL 12 - COST MODEL AND THRESHOLD OPTIMISATION
# Costs are a per-transaction vector, summed - never averaged into a single
# confusion-matrix weight. regime_params() now also accepts liability_share
# and claim_admin overrides for the new sensitivity sweeps.
# =============================================================================


def regime_params(name: str, fp_cost: float | None = None, cap: float | None = None,
                   liability_share: float | None = None, claim_admin: float | None = None) -> dict:
    p = dict(REGIMES[name])
    p["name"] = name
    if fp_cost is not None:
        p["fp_cost"] = float(fp_cost)
    if cap is not None:
        if p["fn_mode"] == "capped_amount":
            p["fn_cap"] = float(cap)
        elif p["fn_mode"] == "flat":
            p["fn_flat"] = float(cap)
    if liability_share is not None:
        p["fn_share"] = float(liability_share)
    if claim_admin is not None:
        p["fn_admin"] = float(claim_admin)
    return p


def outcome_costs(amount: np.ndarray, params: dict) -> tuple:
    a = amount.astype("float64")

    if params["fn_mode"] == "amount":
        c_fn = params["fn_share"] * a
    elif params["fn_mode"] == "capped_amount":
        reimbursable = np.minimum(a, params["fn_cap"])
        net = np.maximum(reimbursable - params["fn_excess"], 0.0)
        c_fn = params["fn_share"] * net + params["fn_admin"]
    elif params["fn_mode"] == "flat":
        c_fn = np.full_like(a, params["fn_flat"])
    else:
        raise ValueError(params["fn_mode"])

    c_fp = np.full_like(a, params["fp_cost"])
    c_tp = np.full_like(a, params["tp_cost"])
    c_tn = np.full_like(a, params["tn_cost"])
    return c_tp, c_fp, c_fn, c_tn


def total_cost(y: np.ndarray, yhat: np.ndarray, amount: np.ndarray, params: dict) -> float:
    c_tp, c_fp, c_fn, c_tn = outcome_costs(amount, params)
    y = y.astype(bool)
    yhat = yhat.astype(bool)
    return float(c_tp[y & yhat].sum() + c_fp[~y & yhat].sum() + c_fn[y & ~yhat].sum() + c_tn[~y & ~yhat].sum())


def build_threshold_grid(p: np.ndarray, size: int) -> np.ndarray:
    q = np.quantile(p, np.linspace(0.0, 1.0, size))
    grid = np.concatenate([q, np.logspace(-7, 0, 300), np.linspace(0.0, 1.0, 201)])
    return np.unique(np.clip(grid, 0.0, 1.0))


def cost_curve(y, p, amount, params, size: int = 1001) -> tuple:
    c_tp, c_fp, c_fn, c_tn = outcome_costs(amount, params)
    y = y.astype(bool)

    o = np.argsort(p, kind="mergesort")
    ps, yb = p[o], y[o]
    cum_fn = np.concatenate([[0.0], np.cumsum(np.where(yb, c_fn[o], 0.0))])
    cum_tn = np.concatenate([[0.0], np.cumsum(np.where(~yb, c_tn[o], 0.0))])
    tot_tp, tot_fp = c_tp[y].sum(), c_fp[~y].sum()
    cum_tp = np.concatenate([[0.0], np.cumsum(np.where(yb, c_tp[o], 0.0))])
    cum_fp = np.concatenate([[0.0], np.cumsum(np.where(~yb, c_fp[o], 0.0))])

    grid = build_threshold_grid(p, size)
    k = np.searchsorted(ps, grid, side="left")
    costs = cum_fn[k] + cum_tn[k] + (tot_tp - cum_tp[k]) + (tot_fp - cum_fp[k])
    return grid, costs


def optimise_threshold(y, p, amount, params, size: int = 1001) -> dict:
    grid, costs = cost_curve(y, p, amount, params, size)
    best = int(np.max(np.flatnonzero(costs == costs.min())))
    return {"threshold": float(grid[best]), "cost": float(costs[best]), "grid": grid, "costs": costs}


def elkan_threshold(amount: np.ndarray, params: dict) -> float:
    c_tp, c_fp, c_fn, c_tn = outcome_costs(amount, params)
    num = c_fp.mean() - c_tn.mean()
    den = num + (c_fn.mean() - c_tp.mean())
    return float(np.clip(num / den, 0.0, 1.0)) if den > 0 else 0.0


def example_dependent_decisions(p: np.ndarray, amount: np.ndarray, params: dict) -> np.ndarray:
    c_tp, c_fp, c_fn, c_tn = outcome_costs(amount, params)
    alert = p * c_tp + (1 - p) * c_fp
    hold = p * c_fn + (1 - p) * c_tn
    return (alert < hold).astype("int8")

In [ ]:
# =============================================================================
# CELL 13 - STATISTICAL SIGNIFICANCE
# =============================================================================


def bootstrap_threshold(y, p, amount, params, n_boot: int, grid_size: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = len(y)
    out = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        out[b] = optimise_threshold(y[idx], p[idx], amount[idx], params, grid_size)["threshold"]
    return out


def bootstrap_cost_at_threshold(y, p, amount, params, threshold: float, n_boot: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = len(y)
    yhat = (p >= threshold).astype("int8")
    out = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        out[b] = total_cost(y[idx], yhat[idx], amount[idx], params) / n
    return out


def compare_regimes_bootstrap(y_val, p_val, amount_val, regime_a: str, regime_b: str,
                               n_boot: int = 500, grid_size: int = 401, seed: int = 42) -> dict:
    params_a, params_b = regime_params(regime_a), regime_params(regime_b)
    rng = np.random.default_rng(seed)
    n = len(y_val)
    thr_a, thr_b = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        thr_a[b] = optimise_threshold(y_val[idx], p_val[idx], amount_val[idx], params_a, grid_size)["threshold"]
        thr_b[b] = optimise_threshold(y_val[idx], p_val[idx], amount_val[idx], params_b, grid_size)["threshold"]
    diff = thr_b - thr_a
    ci = np.percentile(diff, [2.5, 97.5])
    return {
        "regime_a": regime_a, "regime_b": regime_b,
        "threshold_a_mean": float(thr_a.mean()), "threshold_a_ci_lo": float(np.percentile(thr_a, 2.5)), "threshold_a_ci_hi": float(np.percentile(thr_a, 97.5)),
        "threshold_b_mean": float(thr_b.mean()), "threshold_b_ci_lo": float(np.percentile(thr_b, 2.5)), "threshold_b_ci_hi": float(np.percentile(thr_b, 97.5)),
        "diff_mean": float(diff.mean()), "diff_ci_lo": float(ci[0]), "diff_ci_hi": float(ci[1]),
        "significant_at_5pct": bool(ci[0] > 0 or ci[1] < 0),
    }

In [ ]:
# =============================================================================
# CELL 14 - EXPLAINABILITY
# Mean |SHAP value| feature importance for the primary XGBoost model, on the
# same transformed/scaled matrix the model was actually fit on.
# =============================================================================


def compute_shap_importance(model, M_test: np.ndarray, feature_names: list[str], max_rows: int, seed: int) -> tuple:
    import shap
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(M_test), size=min(max_rows, len(M_test)), replace=False)
    sample = M_test[idx]
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(sample)
    if isinstance(sv, list):
        sv = sv[1]
    mean_abs = np.abs(sv).mean(axis=0)
    imp = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs}).sort_values(
        "mean_abs_shap", ascending=False
    ).reset_index(drop=True)
    return imp, sv, sample

In [ ]:
# =============================================================================
# CELL 15 - EVALUATION METRICS
# Accuracy is never reported (meaningless under class imbalance).
# =============================================================================


def threshold_free_metrics(y: np.ndarray, p: np.ndarray) -> dict:
    base = float(y.mean())
    ap = float(average_precision_score(y, p))
    return {
        "auc_pr": ap, "auc_pr_lift": ap / base if base > 0 else np.nan,
        "roc_auc": float(roc_auc_score(y, p)), "brier": float(brier_score_loss(y, p)),
        "ece": expected_calibration_error(y, p), "base_rate": base,
    }


def metrics_at(y, p, amount, threshold: float, params: dict, yhat=None) -> dict:
    yhat = (p >= threshold).astype("int8") if yhat is None else yhat
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    cost = total_cost(y, yhat, amount, params)
    fraud_value = float(amount[y == 1].sum())
    caught_value = float(amount[(y == 1) & (yhat == 1)].sum())
    return {
        "threshold": float(threshold), "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "precision": float(precision_score(y, yhat, zero_division=0)),
        "recall": float(recall_score(y, yhat, zero_division=0)),
        "f1": float(f1_score(y, yhat, zero_division=0)),
        # Reported for completeness only (NEW in v3) - proposal S3.1 explains
        # why accuracy is not a meaningful metric at 3.5% base rate (a model
        # that alerts on nothing scores >96% and catches zero fraud). Do not
        # use this as a headline result; F1/AUC-PR remain the primary metrics.
        "accuracy": float(accuracy_score(y, yhat)),
        "alert_rate": float(yhat.mean()),
        "false_positive_rate": float(fp / max(tn + fp, 1)),
        "value_detection_rate": caught_value / fraud_value if fraud_value > 0 else np.nan,
        "total_cost_gbp": cost, "cost_per_txn_gbp": cost / len(y),
    }


def f1_optimal_threshold(y, p) -> float:
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
    return float(thr[max(int(np.argmax(f1)) - 1, 0)]) if len(thr) else 0.5


def youden_threshold(y, p) -> float:
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[int(np.argmax(tpr - fpr))])


def precision_target_threshold(y, p, target: float) -> tuple[float, bool]:
    """Lowest threshold (on val_thr) whose precision >= target, chosen to
    maximise recall subject to that floor. Returns (threshold, target_met);
    if no threshold reaches target, falls back to the highest precision the
    model actually achieves and reports target_met=False rather than lying
    about which threshold was used."""
    prec, rec, thr = precision_recall_curve(y, p)
    ok = prec[:-1] >= target
    if not ok.any():
        idx = int(np.argmax(prec[:-1])) if len(prec) > 1 else 0
        return (float(thr[idx]) if len(thr) else 0.5), False
    idxs = np.flatnonzero(ok)
    best = idxs[int(np.argmax(rec[idxs]))]
    return float(thr[best]), True


def recall_target_threshold(y, p, target: float) -> tuple[float, bool]:
    """Highest threshold (on val_thr) whose recall >= target, chosen to
    maximise precision subject to that floor. Same honest fallback as
    precision_target_threshold if the target is unreachable."""
    prec, rec, thr = precision_recall_curve(y, p)
    ok = rec[:-1] >= target
    if not ok.any():
        idx = int(np.argmax(rec[:-1])) if len(prec) > 1 else 0
        return (float(thr[idx]) if len(thr) else 0.5), False
    idxs = np.flatnonzero(ok)
    best = idxs[int(np.argmax(prec[idxs]))]
    return float(thr[best]), True


def fit_stack_ensemble(run_id: str, models: tuple, cfg: Config, params: dict) -> dict:
    """Logistic stacker over the calibrated probability streams of `models`
    (NEW in v7). Fit on val_thr - never seen by any base model while
    training; AUC-PR/ROC-AUC are computed purely on TEST, so they are
    leak-safe regardless of any nuance in how the stacker's own threshold
    gets picked. Exploratory, beyond the proposal's 3-separate-models scope -
    see Cell 22."""
    arts = {m: ARTEFACTS[f"{run_id}|{m}"] for m in models}
    y_val, y_test = arts[models[0]]["y_val_thr"], arts[models[0]]["y_test"]
    a_val, a_test = arts[models[0]]["amount_val_thr"], arts[models[0]]["amount_test"]
    Xs_val = np.column_stack([arts[m]["p_val_thr"] for m in models])
    Xs_test = np.column_stack([arts[m]["p_test"] for m in models])

    stacker = LogisticRegression(max_iter=2000, random_state=cfg.random_state)
    stacker.fit(Xs_val, y_val)
    p_val = stacker.predict_proba(Xs_val)[:, 1]
    p_test = stacker.predict_proba(Xs_test)[:, 1]

    tf = threshold_free_metrics(y_test, p_test)
    opt = optimise_threshold(y_val, p_val, a_val, params, cfg.threshold_grid)
    m = metrics_at(y_test, p_test, a_test, opt["threshold"], params)
    weights = dict(zip(models, stacker.coef_[0].tolist()))
    return {**tf, **m, "weights": weights, "intercept": float(stacker.intercept_[0])}

In [ ]:
# =============================================================================
# CELL 16 - FIGURES
# =============================================================================


def _save(fig, stem: str):
    path = FIG_DIR / f"{stem}.png"
    fig.savefig(path)
    plt.close(fig)
    print(f"figure -> {path}")


def plot_pr(curves: dict, base_rate: float, stem: str, title: str):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for label, (y, p) in curves.items():
        prec, rec, _ = precision_recall_curve(y, p)
        ax.plot(rec, prec, lw=1.6, label=f"{label} (AP={average_precision_score(y, p):.3f})")
    ax.axhline(base_rate, ls="--", c="grey", lw=1, label=f"no skill ({base_rate:.3%})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(title)
    ax.legend(fontsize=8); _save(fig, stem)


def plot_roc(curves: dict, stem: str, title: str):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for label, (y, p) in curves.items():
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, lw=1.6, label=f"{label} (AUC={roc_auc_score(y, p):.3f})")
    ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(title); ax.legend(fontsize=8); _save(fig, stem)


def _plot_cost_panel(ax, curves: dict, marks: dict, subtitle: str):
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    keys = list(curves)
    for i, (label, (grid, costs)) in enumerate(curves.items()):
        m = grid > 0
        ax.plot(grid[m], costs[m] / 1e3, lw=1.6, label=label, color=colors[i % len(colors)])
    ax.set_xscale("log")
    ax.set_xlabel("Decision threshold (log scale)")
    ax.set_ylabel("Total cost (GBP thousands)")
    ax.set_title(subtitle)
    if marks:
        y_top = ax.get_ylim()[1]
        for i, (label, t) in enumerate(marks.items()):
            c = colors[keys.index(label) % len(colors)] if label in keys else "black"
            ax.axvline(t, ls=":", lw=1.2, alpha=0.8, color=c)
            ax.annotate(f"{label}\nt={t:.3f}", xy=(t, y_top * (0.92 - 0.14 * i)), fontsize=7, ha="center", color=c)
    ax.legend(fontsize=8)


def plot_cost_by_regime(curves: dict, marks: dict, stem: str, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    small = {k: v for k, v in curves.items() if k != "uk_statutory_cap"}
    _plot_cost_panel(axes[0], small, {k: v for k, v in marks.items() if k in small}, "Literature vs UK regulatory")
    big = {k: v for k, v in curves.items() if k == "uk_statutory_cap"}
    _plot_cost_panel(axes[1], big, {k: v for k, v in marks.items() if k in big}, "Naive flat-cap reading")
    fig.suptitle(title)
    _save(fig, stem)


def plot_calibration(curves: dict, stem: str, title: str, bins: int = 15):
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for label, (y, p) in curves.items():
        frac, mean_p = calibration_curve(y, p, n_bins=bins, strategy="quantile")
        ax.plot(mean_p, frac, "o-", ms=4, lw=1.4, label=f"{label} (ECE={expected_calibration_error(y, p):.4f})")
    ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1, label="perfect")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed fraud rate")
    ax.set_title(title); ax.legend(fontsize=8); _save(fig, stem)


def plot_shap_bar(imp_df: pd.DataFrame, stem: str, title: str, top_n: int = 15):
    top = imp_df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.barh(top["feature"], top["mean_abs_shap"])
    ax.set_xlabel("Mean |SHAP value|"); ax.set_title(title)
    _save(fig, stem)


def plot_threshold_bar(regime_table: pd.DataFrame, stem: str, title: str):
    piv = regime_table.pivot(index="model", columns="regime", values="threshold")
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    piv.plot(kind="bar", ax=ax)
    ax.set_ylabel("Cost-optimal threshold"); ax.set_title(title)
    ax.legend(title="Regime", fontsize=8)
    _save(fig, stem)


def export_table(df: pd.DataFrame, stem: str, caption: str, float_fmt: str = "%.4f"):
    csv_path = TAB_DIR / f"{stem}.csv"
    tex_path = TAB_DIR / f"{stem}.tex"
    df.to_csv(csv_path, index=False)
    tex = df.to_latex(index=False, float_format=float_fmt, caption=caption, label=f"tab:{stem}", escape=True, longtable=False)
    tex_path.write_text(tex)
    print(f"table -> {csv_path.name}, {tex_path.name}")
    return df

In [ ]:
# =============================================================================
# CELL 17 - EXPERIMENT ORCHESTRATOR
# One call = one dataset, one split rule, one cost regime, all requested
# models, tuned hyperparameters, and (optionally) SHAP.
# =============================================================================


def run_experiment(cfg: Config, tag: str = "", verbose: bool = True) -> pd.DataFrame:
    global _LOG
    _LOG = verbose
    t_start = time.time()
    params = regime_params(cfg.regime)
    run_id = tag or f"{cfg.dataset}_{cfg.split}_{cfg.regime}"
    if verbose:
        print(f"\n{'=' * 78}\nRUN {run_id}\n{'=' * 78}")

    X_df, y, amount, order = load_dataset(cfg)
    X_df = engineer_features(X_df, amount, order, cfg)
    sp = make_splits(y, order, cfg)

    prep = FeaturePreparer(cfg).fit(X_df.iloc[sp["train"]])
    M = {k: prep.transform(X_df.iloc[v], scale=True) for k, v in sp.items()}
    Y = {k: y[v] for k, v in sp.items()}
    A = {k: amount[v] for k, v in sp.items()}
    del X_df
    gc.collect()

    M_res, y_res, pi_orig, pi_res = resample_train(M["train"], Y["train"], cfg)

    cfg, hp_traces = tune_hyperparameters(cfg, M_res, y_res, M["val_cal"], Y["val_cal"])
    for _mname, _trace in hp_traces.items():
        TUNING_TRACES[f"{run_id}|{_mname}"] = _trace
    TUNED_CONFIGS[run_id] = cfg   # callers can reuse this cfg so tuning isn't discarded

    rows = []
    for label, yhat in {"approve_everything": np.zeros_like(Y["test"]), "decline_everything": np.ones_like(Y["test"])}.items():
        c = total_cost(Y["test"], yhat, A["test"], params)
        rows.append({
            "run_id": run_id, "dataset": cfg.dataset, "split": cfg.split, "regime": cfg.regime,
            "model": "baseline", "strategy": label, "threshold": np.nan, "total_cost_gbp": c,
            "cost_per_txn_gbp": c / len(Y["test"]), "n_test": len(Y["test"]),
        })
    cost_no_model = rows[0]["total_cost_gbp"]

    for name in cfg.models:
        t_model = time.time()
        model = fit_model(name, cfg, M_res, y_res, M["val_cal"], Y["val_cal"])
        chain = ProbabilityChain(name, model, prep, cfg, pi_orig, pi_res)

        raw = {k: chain.model.predict_proba(M[k])[:, 1].astype("float64") for k in ("val_cal", "val_thr", "test")}
        chain.fit_recalibrator(raw["val_cal"], Y["val_cal"])
        P = {k: chain.finalise(v) for k, v in raw.items()}

        tf = threshold_free_metrics(Y["test"], P["test"])

        opt = optimise_threshold(Y["val_thr"], P["val_thr"], A["val_thr"], params, cfg.threshold_grid)
        _prec_key = f"precision_target_{int(cfg.precision_target * 100)}"
        _rec_key = f"recall_target_{int(cfg.recall_target * 100)}"
        _prec_thr, _prec_met = precision_target_threshold(Y["val_thr"], P["val_thr"], cfg.precision_target)
        _rec_thr, _rec_met = recall_target_threshold(Y["val_thr"], P["val_thr"], cfg.recall_target)

        strategies = {
            "default_0.5": 0.5, "cost_optimal": opt["threshold"],
            "f1_optimal": f1_optimal_threshold(Y["val_thr"], P["val_thr"]),
            "youden_j": youden_threshold(Y["val_thr"], P["val_thr"]),
            "elkan_closed_form": elkan_threshold(A["val_thr"], params),
            _prec_key: _prec_thr, _rec_key: _rec_thr,
        }
        target_met = {_prec_key: _prec_met, _rec_key: _rec_met}

        for strat, thr in strategies.items():
            m = metrics_at(Y["test"], P["test"], A["test"], thr, params)
            rows.append({
                "run_id": run_id, "dataset": cfg.dataset, "split": cfg.split, "regime": cfg.regime,
                "model": name, "strategy": strat, "n_test": len(Y["test"]),
                "fit_seconds": round(time.time() - t_model, 1),
                "target_met": target_met.get(strat, np.nan), **tf, **m,
            })

        yhat_ed = example_dependent_decisions(P["test"], A["test"], params)
        m = metrics_at(Y["test"], P["test"], A["test"], np.nan, params, yhat=yhat_ed)
        m["threshold"] = np.nan
        rows.append({
            "run_id": run_id, "dataset": cfg.dataset, "split": cfg.split, "regime": cfg.regime,
            "model": name, "strategy": "example_dependent", "n_test": len(Y["test"]), **tf, **m,
        })

        ARTEFACTS[f"{run_id}|{name}"] = {
            "y_test": Y["test"], "p_test": P["test"], "amount_test": A["test"],
            "y_val_thr": Y["val_thr"], "p_val_thr": P["val_thr"], "amount_val_thr": A["val_thr"],
            "val_grid": opt["grid"], "val_costs": opt["costs"], "cost_optimal_threshold": opt["threshold"],
            "feature_names": prep.feature_names_,
        }

        if cfg.run_shap and name == "xgboost":
            imp, sv, sample = compute_shap_importance(model, M["test"], prep.feature_names_, cfg.shap_max_rows, cfg.random_state)
            SHAP_RESULTS[f"{run_id}|{name}"] = {"importance": imp, "shap_values": sv, "sample": sample}
            log(f"shap: computed importance for {len(imp)} features on {len(sample):,} rows")

        del model, raw, P
        gc.collect()

    df = pd.DataFrame(rows)
    base = df[df.strategy == "default_0.5"].set_index("model")["total_cost_gbp"]
    df["saving_vs_default_gbp"] = df.apply(lambda r: base.get(r["model"], np.nan) - r["total_cost_gbp"], axis=1)
    df["saving_vs_default_pct"] = 100 * df["saving_vs_default_gbp"] / df["model"].map(base)
    df["saving_vs_no_model_pct"] = 100 * (cost_no_model - df["total_cost_gbp"]) / cost_no_model

    RESULTS.append(df)
    if verbose:
        cols = ["model", "strategy", "threshold", "recall", "precision", "f1", "alert_rate", "cost_per_txn_gbp", "saving_vs_default_pct"]
        print(df[[c for c in cols if c in df.columns]].to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
        print(f"run completed in {time.time() - t_start:.1f}s")
    _LOG = True
    return df


def all_results() -> pd.DataFrame:
    return pd.concat(RESULTS, ignore_index=True) if RESULTS else pd.DataFrame()

In [ ]:
# =============================================================================
# CELL 18 - SMOKE TEST
# Runs every code path (incl. hyperparameter search, bootstrap CI, SHAP) on
# generated data in well under a minute, no download needed. Run this first
# after any edit. Nothing produced here may appear in the dissertation.
# =============================================================================

smoke_cfg = replace(
    CFG, dataset="synthetic", models=("logistic", "random_forest", "xgboost"), regime="uk_regulatory",
    rf_n_estimators=60, xgb_n_estimators=200, tune_hyperparams=True, hp_search_iter=3,
    xgb_search_n_estimators=80, run_shap=True, shap_max_rows=300, bootstrap_n=40, bootstrap_grid=101,
)

_smoke = run_experiment(smoke_cfg, tag="smoke")

_chk = _smoke[_smoke.model != "baseline"].pivot_table(index="model", columns="strategy", values="cost_per_txn_gbp")
print("\ncost per transaction, GBP")
print(_chk[["default_0.5", "cost_optimal", "f1_optimal", "example_dependent"]].to_string(float_format=lambda v: f"{v:,.4f}"))
print("\nECE:", _smoke[_smoke.model != "baseline"].groupby("model")["ece"].first().round(5).to_dict())

_smoke_art = ARTEFACTS["smoke|xgboost"]
_smoke_cmp = compare_regimes_bootstrap(
    _smoke_art["y_val_thr"], _smoke_art["p_val_thr"], _smoke_art["amount_val_thr"],
    "literature", "uk_regulatory", n_boot=smoke_cfg.bootstrap_n, grid_size=smoke_cfg.bootstrap_grid,
)
print("\nbootstrap regime comparison (smoke):", _smoke_cmp)

if "smoke|xgboost" in SHAP_RESULTS:
    print("\ntop-5 SHAP features (smoke):")
    print(SHAP_RESULTS["smoke|xgboost"]["importance"].head())
    plot_shap_bar(SHAP_RESULTS["smoke|xgboost"]["importance"], "smoke_check_shap", "SHAP smoke check")

plot_cost_by_regime(
    {"literature": (_smoke_art["val_grid"], _smoke_art["val_costs"])},
    {"literature": _smoke_art["cost_optimal_threshold"]},
    "smoke_check_cost_curve", "Cost curve smoke check",
)

print("\nimbalance-method code-path check (smoke, XGBoost only):")
for _m in ("smote", "borderline_smote", "smote_tomek", "class_weight", "none"):
    _imb_cfg = replace(smoke_cfg, models=("xgboost",), imbalance_method=_m, tune_hyperparams=False, run_shap=False)
    _imb_r = run_experiment(_imb_cfg, tag=f"smoke_imb_{_m}", verbose=False)
    _imb_row = _imb_r[(_imb_r.model == "xgboost") & (_imb_r.strategy == "default_0.5")].iloc[0]
    print(f"  {_m:<16} auc_pr={_imb_row['auc_pr']:.4f}")

_smoke_stack = fit_stack_ensemble("smoke", smoke_cfg.models, smoke_cfg, regime_params(smoke_cfg.regime))
print(f"\nstacked ensemble check (smoke): AUC-PR={_smoke_stack['auc_pr']:.4f} weights={_smoke_stack['weights']}")

RESULTS.clear()
ARTEFACTS.clear()
TUNING_TRACES.clear()
SHAP_RESULTS.clear()


RUN smoke
synthetic: 40,000 rows x 21 cols | fraud 3.0225% (1,209) | amount GBP mean 71.61 median 37.94 max 3,368.49
train     n=  28,000  fraud=3.0214%  positives=   846
val_cal   n=   2,999  fraud=3.0010%  positives=    90
val_thr   n=   3,000  fraud=3.0333%  positives=    91
test      n=   6,001  fraud=3.0328%  positives=   182
prepared: kept 28/29 cols (27 numeric, 1 categorical), dropped 1, aggregate groups on: []
smote: 28,000 -> 29,869 rows | prior 3.0214% -> 9.0897%
  rf [1/3] {'n_estimators': 150, 'max_depth': 22, 'min_samples_leaf': 5} -> AUC-PR=0.2856 (64.3s)
  rf [2/3] {'n_estimators': 200, 'max_depth': 14, 'min_samples_leaf': 8} -> AUC-PR=0.2648 (39.0s)
  rf [3/3] {'n_estimators': 150, 'max_depth': 18, 'min_samples_leaf': 3} -> AUC-PR=0.2638 (31.8s)
random_forest tuned: best AUC-PR(val_cal)=0.2856 params={'n_estimators': 150, 'max_depth': 22, 'min_samples_leaf': 5}
  xgb [1/3] {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8} -> AUC-PR=0.28

In [ ]:
# =============================================================================
# CELL 19 - IMBALANCE METHOD COMPARISON
# Rather than assume SMOTE is best, this compares it against Borderline
# -SMOTE and class-weighting empirically, cheaply, on the tuning subsample -
# XGBoost only, default hyperparameters, one fit each - and carries the
# winner into Cell 20's search and Cell 21's full-data fit.
#
# SMOTE-Tomek is deliberately EXCLUDED from this automatic comparison.
# Tomek-link cleaning is an O(n log n) nearest-neighbour search over the
# whole resampled set in ~440 dimensions, and dimensionality this high pushes
# even efficient neighbour search close to brute-force cost. It is fully
# wired up (Config.imbalance_method="smote_tomek") if you want to try it
# yourself - time it on TUNING_SAMPLE_ROWS before ever pointing it at the
# full 590k-row primary run.
# =============================================================================

IMBALANCE_METHODS_TO_COMPARE = ("smote", "borderline_smote", "class_weight")

imbalance_rows = []
for _method in IMBALANCE_METHODS_TO_COMPARE:
    _imb_cfg = replace(
        CFG, dataset="ieee", split="stratified", regime="uk_regulatory",
        sample_rows=TUNING_SAMPLE_ROWS, models=("xgboost",),
        imbalance_method=_method, tune_hyperparams=False, run_shap=False,
    )
    _t0 = time.time()
    _df = run_experiment(_imb_cfg, tag=f"imb_{_method}", verbose=False)
    _elapsed = time.time() - _t0
    _row = _df[(_df.model == "xgboost") & (_df.strategy == "default_0.5")].iloc[0]
    imbalance_rows.append({
        "imbalance_method": _method, "auc_pr": _row["auc_pr"], "roc_auc": _row["roc_auc"],
        "precision": _row["precision"], "recall": _row["recall"], "wall_seconds": round(_elapsed, 1),
    })
    print(f"  {_method:<16} auc_pr={_row['auc_pr']:.4f} roc_auc={_row['roc_auc']:.4f} ({_elapsed:.0f}s)")

imbalance_table = pd.DataFrame(imbalance_rows).sort_values("auc_pr", ascending=False).reset_index(drop=True)
print("\nimbalance-handling method comparison (XGBoost, tuning subsample, default hyperparameters)")
print(imbalance_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

BEST_IMBALANCE_METHOD = str(imbalance_table.iloc[0]["imbalance_method"])
print(f"\nselected imbalance_method = {BEST_IMBALANCE_METHOD!r} for the tuning search and primary run")

  smote            auc_pr=0.6565 roc_auc=0.9343 (448s)
  borderline_smote auc_pr=0.6646 roc_auc=0.9339 (485s)
  class_weight     auc_pr=0.6695 roc_auc=0.9265 (388s)

imbalance-handling method comparison (XGBoost, tuning subsample, default hyperparameters)
imbalance_method  auc_pr  roc_auc  precision  recall  wall_seconds
    class_weight  0.6695   0.9265     0.8133  0.5584      388.0000
borderline_smote  0.6646   0.9339     0.8182  0.5368      484.7000
           smote  0.6565   0.9343     0.8232  0.5317      447.6000

selected imbalance_method = 'class_weight' for the tuning search and primary run


In [ ]:
# =============================================================================
# CELL 20 - HYPERPARAMETER SEARCH (on a subsample, not the full dataset)
# =============================================================================

tuning_cfg = replace(
    CFG, dataset="ieee", split="stratified", regime="uk_regulatory",
    models=("logistic", "random_forest", "xgboost"), sample_rows=TUNING_SAMPLE_ROWS,
    imbalance_method=BEST_IMBALANCE_METHOD, tune_hyperparams=True, hp_search_iter=12, run_shap=False,
)

_tuning_run = run_experiment(tuning_cfg, tag="tuning_search")
tuned_cfg = TUNED_CONFIGS.get("tuning_search", tuning_cfg)
print("\ntuned architecture (from the subsample search, applied to Cell 20's full-data fit):")
for _f in ("rf_n_estimators", "rf_max_depth", "rf_min_samples_leaf",
           "xgb_max_depth", "xgb_learning_rate", "xgb_subsample", "xgb_colsample_bytree", "lr_C"):
    print(f"  {_f} = {getattr(tuned_cfg, _f)}")


RUN tuning_search
ieee: 150,000 rows x 432 cols | fraud 3.4993% (5,249) | amount GBP mean 106.61 median 54.47 max 25,230.54
train     n= 105,000  fraud=3.4990%  positives= 3,674
val_cal   n=  11,249  fraud=3.4936%  positives=   393
val_thr   n=  11,250  fraud=3.5022%  positives=   394
test      n=  22,501  fraud=3.5021%  positives=   788
prepared: kept 433/446 cols (401 numeric, 32 categorical), dropped 13, aggregate groups on: [('card1',), ('addr1',), ('card1', 'addr1'), ('card2',), ('P_emaildomain',), ('card1', 'P_emaildomain')]
imbalance_method=class_weight: no resampling | prior 3.4990%
  rf [1/12] {'n_estimators': 150, 'max_depth': 22, 'min_samples_leaf': 5} -> AUC-PR=0.5629 (83.5s)
  rf [2/12] {'n_estimators': 200, 'max_depth': 14, 'min_samples_leaf': 8} -> AUC-PR=0.5336 (92.4s)
  rf [3/12] {'n_estimators': 150, 'max_depth': 18, 'min_samples_leaf': 3} -> AUC-PR=0.5549 (82.6s)
  rf [4/12] {'n_estimators': 150, 'max_depth': 18, 'min_samples_leaf': 8} -> AUC-PR=0.5482 (81.6s)
  rf 

In [ ]:
# =============================================================================
# CELL 21 - PRIMARY RUN (full dataset, tuned architecture)
# =============================================================================

primary_cfg = replace(
    tuned_cfg, sample_rows=None, tune_hyperparams=False, run_shap=True,
)

primary = run_experiment(primary_cfg, tag="primary")


RUN primary
ieee: 590,540 rows x 432 cols | fraud 3.4990% (20,663) | amount GBP mean 106.67 median 54.33 max 25,230.54
train     n= 413,378  fraud=3.4990%  positives=14,464
val_cal   n=  44,290  fraud=3.4997%  positives= 1,550
val_thr   n=  44,290  fraud=3.4974%  positives= 1,549
test      n=  88,582  fraud=3.4996%  positives= 3,100
prepared: kept 434/446 cols (402 numeric, 32 categorical), dropped 12, aggregate groups on: [('card1',), ('addr1',), ('card1', 'addr1'), ('card2',), ('P_emaildomain',), ('card1', 'P_emaildomain')]
imbalance_method=class_weight: no resampling | prior 3.4990%
logistic: fitted in 219.0s
random_forest: fitted in 760.9s
xgboost: fitted in 1924.4s | best_iteration=1999
shap: computed importance for 452 features on 2,000 rows
        model            strategy  threshold  recall  precision     f1  alert_rate  cost_per_txn_gbp  saving_vs_default_pct
     baseline  approve_everything        NaN     NaN        NaN    NaN         NaN            3.6128                 

In [ ]:
# =============================================================================
# CELL 22 - STACKED ENSEMBLE (exploratory - beyond proposal scope)
# =============================================================================

stack_result = fit_stack_ensemble("primary", primary_cfg.models, primary_cfg, regime_params(primary_cfg.regime))
_best_single_auc_pr = max(
    threshold_free_metrics(ARTEFACTS[f"primary|{m}"]["y_test"], ARTEFACTS[f"primary|{m}"]["p_test"])["auc_pr"]
    for m in primary_cfg.models
)
print("\nstacked ensemble (logistic combiner over Logistic Regression + Random Forest + XGBoost)")
print(f"  weights: { {k: round(v, 4) for k, v in stack_result['weights'].items()} }  intercept: {stack_result['intercept']:.4f}")
print(f"  AUC-PR={stack_result['auc_pr']:.4f}  ROC-AUC={stack_result['roc_auc']:.4f}  "
      f"precision={stack_result['precision']:.4f}  recall={stack_result['recall']:.4f}  "
      f"F1={stack_result['f1']:.4f}  cost/txn=GBP{stack_result['cost_per_txn_gbp']:.4f}")
print(f"  best single model AUC-PR = {_best_single_auc_pr:.4f}  |  "
      f"stack {'beats' if stack_result['auc_pr'] > _best_single_auc_pr else 'does not beat'} it "
      f"by {stack_result['auc_pr'] - _best_single_auc_pr:+.4f}")

stack_table = pd.DataFrame([{
    "auc_pr": stack_result["auc_pr"], "auc_pr_lift": stack_result["auc_pr_lift"], "roc_auc": stack_result["roc_auc"],
    "threshold": stack_result["threshold"], "precision": stack_result["precision"], "recall": stack_result["recall"],
    "f1": stack_result["f1"], "accuracy": stack_result["accuracy"], "cost_per_txn_gbp": stack_result["cost_per_txn_gbp"],
    "weight_logistic": stack_result["weights"].get("logistic"), "weight_random_forest": stack_result["weights"].get("random_forest"),
    "weight_xgboost": stack_result["weights"].get("xgboost"), "intercept": stack_result["intercept"],
    "best_single_model_auc_pr": _best_single_auc_pr, "beats_best_single_model": stack_result["auc_pr"] > _best_single_auc_pr,
}])


stacked ensemble (logistic combiner over Logistic Regression + Random Forest + XGBoost)
  weights: {'logistic': -0.2041, 'random_forest': -0.922, 'xgboost': 9.6686}  intercept: -4.9767
  AUC-PR=0.8714  ROC-AUC=0.9620  precision=0.8014  recall=0.8303  F1=0.8156  cost/txn=GBP1.1748
  best single model AUC-PR = 0.8686  |  stack beats it by +0.0028


In [ ]:
# =============================================================================
# CELL 23 - SAMPLE-SIZE ABLATION
# XGBoost only, untuned, at increasing sample sizes up to the full dataset -
# quantifies what a subsample costs relative to the primary run above.
# =============================================================================

ablation_rows = []
for _n in SAMPLE_SIZE_ABLATION:
    _cfg = replace(primary_cfg, sample_rows=_n, models=("xgboost",), tune_hyperparams=False, run_shap=False)
    _t0 = time.time()
    _df = run_experiment(_cfg, tag=f"ablation_{_n or 'full'}", verbose=False)
    _elapsed = time.time() - _t0
    _row = _df[(_df.strategy == "cost_optimal") & (_df.model == "xgboost")].iloc[0]
    ablation_rows.append({
        "sample_rows": _n or "full (590,540)", "auc_pr": _row["auc_pr"], "threshold": _row["threshold"],
        "recall": _row["recall"], "precision": _row["precision"], "cost_per_txn_gbp": _row["cost_per_txn_gbp"],
        "wall_seconds": round(_elapsed, 1),
    })

sample_size_table = pd.DataFrame(ablation_rows)
print("\nsample-size ablation")
print(sample_size_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))


sample-size ablation
   sample_rows  auc_pr  threshold  recall  precision  cost_per_txn_gbp  wall_seconds
         50000  0.5079     0.3571  0.5285     0.6495            2.2972      135.1000
        100000  0.6277     0.4918  0.5600     0.7406            2.2588      142.3000
        200000  0.7513     0.2748  0.7190     0.7498            1.7056      853.8000
full (590,540)  0.8686     0.3958  0.7865     0.8827            1.2065    2,027.5000


In [ ]:
# =============================================================================
# CELL 24 - COST REGIME COMPARISON
# =============================================================================

regime_runs = {}
for _regime in ("literature", "uk_regulatory", "uk_statutory_cap"):
    _params = regime_params(_regime)
    _rows = []
    for _model in primary_cfg.models:
        _src = ARTEFACTS[f"primary|{_model}"]
        _opt = optimise_threshold(_src["y_val_thr"], _src["p_val_thr"], _src["amount_val_thr"], _params, CFG.threshold_grid)
        _m = metrics_at(_src["y_test"], _src["p_test"], _src["amount_test"], _opt["threshold"], _params)
        _c_default = total_cost(_src["y_test"], (_src["p_test"] >= 0.5).astype("int8"), _src["amount_test"], _params)
        _rows.append({
            "run_id": f"regime_{_regime}", "dataset": primary_cfg.dataset, "split": primary_cfg.split,
            "regime": _regime, "model": _model, "strategy": "cost_optimal",
            "n_test": len(_src["y_test"]), **_m,
            "saving_vs_default_pct": 100 * (_c_default - _m["total_cost_gbp"]) / _c_default,
        })
        ARTEFACTS[f"regime_{_regime}|{_model}"] = {
            "y_test": _src["y_test"], "p_test": _src["p_test"], "amount_test": _src["amount_test"],
            "y_val_thr": _src["y_val_thr"], "p_val_thr": _src["p_val_thr"], "amount_val_thr": _src["amount_val_thr"],
            "val_grid": _opt["grid"], "val_costs": _opt["costs"], "cost_optimal_threshold": _opt["threshold"],
            "feature_names": _src["feature_names"],
        }
    regime_runs[_regime] = pd.DataFrame(_rows)
    print(f"\n{_regime}")
    print(regime_runs[_regime][["model", "threshold", "recall", "precision", "alert_rate", "cost_per_txn_gbp", "saving_vs_default_pct"]].to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

regime_table = pd.concat(regime_runs.values(), ignore_index=True)


literature
        model  threshold  recall  precision  alert_rate  cost_per_txn_gbp  saving_vs_default_pct
     logistic     0.0501  0.6890     0.1830      0.1318            2.4418                29.8253
random_forest     0.0749  0.7719     0.3680      0.0734            1.7016                31.4397
      xgboost     0.0601  0.8823     0.5475      0.0564            0.9848                23.8877

uk_regulatory
        model  threshold  recall  precision  alert_rate  cost_per_txn_gbp  saving_vs_default_pct
     logistic     0.2321  0.3829     0.5145      0.0260            2.9264                 4.0960
random_forest     0.3593  0.5913     0.7160      0.0289            2.0541                 8.0842
      xgboost     0.3958  0.7865     0.8827      0.0312            1.2065                 2.4420

uk_statutory_cap
        model  threshold  recall  precision  alert_rate  cost_per_txn_gbp  saving_vs_default_pct
     logistic     0.0000  1.0000     0.0350      1.0000           34.0551         

In [ ]:
# =============================================================================
# CELL 25 - ROBUSTNESS
# A. Temporal split.  B. ULB Credit Card Fraud, secondary dataset (different
# continent/decade, 0.17% base rate vs IEEE's 3.5%).  C. PaySim, THIRD
# dataset ( - mobile-money push-payment fraud, the closest public
# proxy to APP fraud.
# =============================================================================

temporal = run_experiment(replace(primary_cfg, split="temporal", tune_hyperparams=False, run_shap=False), tag="temporal")

ulb = run_experiment(
    replace(primary_cfg, dataset="ulb", sample_rows=None, smote_ratio=0.05, tune_hyperparams=False, run_shap=False),
    tag="ulb",
)

paysim = None
if list(DATA_DIR.glob("PS_2017*.csv")) or (DATA_DIR / "paysim.csv").exists():
    paysim = run_experiment(
        replace(primary_cfg, dataset="paysim", sample_rows=300_000, smote_ratio=0.05, tune_hyperparams=False, run_shap=False),
        tag="paysim",
    )
else:
    print("PaySim CSV not found in DATA_DIR - skipping (set FETCH_PAYSIM=True in Cell 3; see README_v7.md)")

seed_rows = []
for _seed in (0, 1, 2, 42, 2024):
    _df = run_experiment(
        replace(primary_cfg, random_state=_seed, models=("xgboost",), sample_rows=primary_cfg.seed_stability_sample, tune_hyperparams=False, run_shap=False),
        tag=f"seed_{_seed}", verbose=False,
    )
    _df = _df[(_df.strategy == "cost_optimal") & (_df.model == "xgboost")]
    seed_rows.append({"seed": _seed, **_df.iloc[0][["threshold", "recall", "precision", "cost_per_txn_gbp"]].to_dict()})

seed_table = pd.DataFrame(seed_rows)
print("\nthreshold stability across seeds")
print(seed_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print(f"\nthreshold mean {seed_table.threshold.mean():.4f} sd {seed_table.threshold.std():.4f} cv {seed_table.threshold.std() / seed_table.threshold.mean():.3f}")


RUN temporal
ieee: 590,540 rows x 432 cols | fraud 3.4990% (20,663) | amount GBP mean 106.67 median 54.33 max 25,230.54
train     n= 413,378  fraud=3.5169%  positives=14,538
val_cal   n=  44,290  fraud=3.4342%  positives= 1,521
val_thr   n=  44,291  fraud=3.4341%  positives= 1,521
test      n=  88,581  fraud=3.4804%  positives= 3,083
prepared: kept 433/446 cols (401 numeric, 32 categorical), dropped 13, aggregate groups on: [('card1',), ('addr1',), ('card1', 'addr1'), ('card2',), ('P_emaildomain',), ('card1', 'P_emaildomain')]
imbalance_method=class_weight: no resampling | prior 3.5169%
logistic: fitted in 203.6s
random_forest: fitted in 785.1s
xgboost: fitted in 1859.2s | best_iteration=1998
        model            strategy  threshold  recall  precision     f1  alert_rate  cost_per_txn_gbp  saving_vs_default_pct
     baseline  approve_everything        NaN     NaN        NaN    NaN         NaN            3.6603                    NaN
     baseline  decline_everything        NaN     

In [ ]:
# =============================================================================
# CELL 26 - SENSITIVITY ANALYSIS
# A. FCA false-positive proxy.  B. PSR cap.  C. APP amount uplift.
# D/E. Liability share and claim-admin cost
# =============================================================================

SWEEP_KEY = "primary|xgboost"


def sweep(key: str, regime: str, values, kind: str) -> pd.DataFrame:
    art = ARTEFACTS[key]
    out = []
    for v in values:
        a_val, a_test = art["amount_val_thr"], art["amount_test"]
        if kind == "fp_cost":
            params = regime_params(regime, fp_cost=v)
        elif kind == "cap":
            params = regime_params(regime, cap=v)
        elif kind == "liability_share":
            params = regime_params(regime, liability_share=v)
        elif kind == "claim_admin":
            params = regime_params(regime, claim_admin=v)
        elif kind == "app_uplift":
            params = regime_params(regime)
            a_val, a_test = a_val * v, a_test * v
        else:
            raise ValueError(kind)

        opt = optimise_threshold(art["y_val_thr"], art["p_val_thr"], a_val, params, CFG.threshold_grid)
        m = metrics_at(art["y_test"], art["p_test"], a_test, opt["threshold"], params)
        c_default = total_cost(art["y_test"], (art["p_test"] >= 0.5).astype("int8"), a_test, params)
        out.append({
            kind: v, "regime": regime, "threshold": opt["threshold"], "recall": m["recall"], "precision": m["precision"],
            "f1": m["f1"], "alert_rate": m["alert_rate"], "value_detection_rate": m["value_detection_rate"],
            "cost_per_txn_gbp": m["cost_per_txn_gbp"], "saving_vs_default_pct": 100 * (c_default - m["total_cost_gbp"]) / c_default,
        })
    return pd.DataFrame(out)


sweep_fp = sweep(SWEEP_KEY, "uk_regulatory", FP_COST_SWEEP, "fp_cost")
sweep_cap = sweep(SWEEP_KEY, "uk_regulatory", CAP_SWEEP, "cap")
sweep_app = sweep(SWEEP_KEY, "uk_regulatory", APP_UPLIFT_SWEEP, "app_uplift")
sweep_liability = sweep(SWEEP_KEY, "uk_regulatory", LIABILITY_SHARE_SWEEP, "liability_share")
sweep_admin = sweep(SWEEP_KEY, "uk_regulatory", CLAIM_ADMIN_SWEEP, "claim_admin")

for _name, _t in [("false-positive cost", sweep_fp), ("PSR cap", sweep_cap), ("APP uplift", sweep_app),
                   ("liability share", sweep_liability), ("claim admin cost", sweep_admin)]:
    print(f"\n{_name}")
    print(_t.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))


false-positive cost
 fp_cost        regime  threshold  recall  precision     f1  alert_rate  value_detection_rate  cost_per_txn_gbp  saving_vs_default_pct
 10.0000 uk_regulatory     0.0870  0.8677     0.6292 0.7295      0.0483                0.8414            0.9534                18.8145
 25.0000 uk_regulatory     0.2254  0.8184     0.8248 0.8216      0.0347                0.7793            1.1170                 7.8203
 35.0000 uk_regulatory     0.3958  0.7865     0.8827 0.8318      0.0312                0.7439            1.2065                 2.4420
 50.0000 uk_regulatory     0.4000  0.7774     0.8972 0.8330      0.0303                0.7309            1.2724                 0.1314
 75.0000 uk_regulatory     0.4000  0.7774     0.8972 0.8330      0.0303                0.7309            1.3503                -1.0362
100.0000 uk_regulatory     0.4000  0.7774     0.8972 0.8330      0.0303                0.7309            1.4282                -2.0997
120.0000 uk_regulatory     0.5833 

In [ ]:
# =============================================================================
# CELL 27 - STATISTICAL SIGNIFICANCE TESTING
# =============================================================================

significance_rows = []
for _model in primary_cfg.models:
    _key = f"regime_uk_regulatory|{_model}"
    if _key not in ARTEFACTS:
        continue
    _art = ARTEFACTS[_key]
    _cmp = compare_regimes_bootstrap(
        _art["y_val_thr"], _art["p_val_thr"], _art["amount_val_thr"],
        "literature", "uk_regulatory", n_boot=CFG.bootstrap_n, grid_size=CFG.bootstrap_grid,
        seed=CFG.random_state,
    )
    _cmp["model"] = MODEL_LABELS.get(_model, _model)
    significance_rows.append(_cmp)

significance_table = pd.DataFrame(significance_rows)[
    ["model", "regime_a", "regime_b", "threshold_a_mean", "threshold_a_ci_lo", "threshold_a_ci_hi",
     "threshold_b_mean", "threshold_b_ci_lo", "threshold_b_ci_hi", "diff_mean", "diff_ci_lo", "diff_ci_hi", "significant_at_5pct"]
]
print("\nbootstrap significance: literature vs uk_regulatory threshold")
print(significance_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

# Realised-cost CI at the fixed cost-optimal threshold, primary model.
_cost_ci_rows = []
for _regime in ("literature", "uk_regulatory", "uk_statutory_cap"):
    _key = f"regime_{_regime}|xgboost"
    if _key not in ARTEFACTS:
        continue
    _art = ARTEFACTS[_key]
    _params = regime_params(_regime)
    _costs = bootstrap_cost_at_threshold(_art["y_test"], _art["p_test"], _art["amount_test"], _params,
                                          _art["cost_optimal_threshold"], CFG.bootstrap_n, CFG.random_state)
    _cost_ci_rows.append({
        "regime": _regime, "threshold": _art["cost_optimal_threshold"], "cost_per_txn_mean": _costs.mean(),
        "cost_per_txn_ci_lo": np.percentile(_costs, 2.5), "cost_per_txn_ci_hi": np.percentile(_costs, 97.5),
    })
cost_ci_table = pd.DataFrame(_cost_ci_rows)
print("\nbootstrap cost CI at fixed cost-optimal threshold (XGBoost)")
print(cost_ci_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))


bootstrap significance: literature vs uk_regulatory threshold
              model   regime_a      regime_b  threshold_a_mean  threshold_a_ci_lo  threshold_a_ci_hi  threshold_b_mean  threshold_b_ci_lo  threshold_b_ci_hi  diff_mean  diff_ci_lo  diff_ci_hi  significant_at_5pct
Logistic Regression literature uk_regulatory            0.0741             0.0501             0.1133            0.2702             0.2131             0.3756     0.1960      0.1006      0.3255                 True
      Random Forest literature uk_regulatory            0.0750             0.0450             0.0859            0.2992             0.1800             0.3593     0.2242      0.1039      0.3125                 True
            XGBoost literature uk_regulatory            0.0711             0.0383             0.0928            0.3402             0.2250             0.4050     0.2691      0.1322      0.3617                 True

bootstrap cost CI at fixed cost-optimal threshold (XGBoost)
          regime  thresh

In [ ]:
# =============================================================================
# CELL 28 - SHAP EXPLAINABILITY
# =============================================================================

if "primary|xgboost" in SHAP_RESULTS:
    shap_importance = SHAP_RESULTS["primary|xgboost"]["importance"]
    print("\ntop-15 SHAP features, primary XGBoost model")
    print(shap_importance.head(15).to_string(index=False, float_format=lambda v: f"{v:,.5f}"))
else:
    print("SHAP not computed for the primary run (set primary_cfg.run_shap=True and re-run Cell 20)")
    shap_importance = pd.DataFrame(columns=["feature", "mean_abs_shap"])


top-15 SHAP features, primary XGBoost model
                         feature  mean_abs_shap
                             C13        0.55119
                  TransactionAmt        0.47841
                              C1        0.38375
                             C14        0.33597
           fe_card1_count_so_far        0.31290
                     fe_hour_sin        0.30117
                           card1        0.28452
                    fe_dayofweek        0.27576
        fe_card1_addr1_amt_count        0.27291
                             V70        0.26705
         fe_card1_addr1_amt_mean        0.26628
fe_card1_P_emaildomain_amt_count        0.25895
                             C11        0.25087
               fe_amount_decimal        0.24637
              fe_card1_amt_count        0.24267


In [ ]:
# =============================================================================
# CELL 29 - FIGURES AND TABLES
# =============================================================================

_pr_curves, _cal_curves = {}, {}
for _m in primary_cfg.models:
    _k = f"primary|{_m}"
    if _k in ARTEFACTS:
        _a = ARTEFACTS[_k]
        _pr_curves[MODEL_LABELS[_m]] = (_a["y_test"], _a["p_test"])
        _cal_curves[MODEL_LABELS[_m]] = (_a["y_test"], _a["p_test"])

_base = float(ARTEFACTS[f"primary|{primary_cfg.models[0]}"]["y_test"].mean())
plot_pr(_pr_curves, _base, "fig1_precision_recall", "Precision-recall, IEEE-CIS test set")
plot_roc(_pr_curves, "fig2_roc", "ROC, IEEE-CIS test set")
plot_calibration(_cal_curves, "fig3_calibration", "Reliability after prior correction and isotonic recalibration")

_cc, _marks = {}, {}
for _regime in ("literature", "uk_regulatory", "uk_statutory_cap"):
    _k = f"regime_{_regime}|xgboost"
    if _k in ARTEFACTS:
        _a = ARTEFACTS[_k]
        _cc[_regime] = (_a["val_grid"], _a["val_costs"])
        _marks[_regime] = _a["cost_optimal_threshold"]
plot_cost_by_regime(_cc, _marks, "fig4_cost_by_regime", "Total cost against threshold by cost regime (XGBoost, validation)")

fig, axes = plt.subplots(1, 5, figsize=(21, 4))
_panels = [
    (sweep_fp, "fp_cost", "False-positive cost (GBP)", False, "FCA Consumer Duty proxy"),
    (sweep_cap, "cap", "PSR reimbursement cap (GBP)", True, "PSR PS23/3 cap"),
    (sweep_app, "app_uplift", "APP amount uplift factor", False, "APP loss scenario"),
    (sweep_liability, "liability_share", "Sending-PSP liability share", False, "PS23/3 liability split"),
    (sweep_admin, "claim_admin", "Claim admin cost (GBP)", False, "Claim-handling cost"),
]
for ax, (t, xcol, xlabel, logx, sub) in zip(axes, _panels):
    ax.plot(t[xcol], t["threshold"], "o-", lw=1.6)
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("Cost-optimal threshold"); ax.set_title(sub)
_save(fig, "fig5_sensitivity")

if not shap_importance.empty:
    plot_shap_bar(shap_importance, "fig6_shap_importance", "Mean |SHAP value|, primary XGBoost model")

plot_threshold_bar(regime_table[regime_table.model != "baseline"], "fig7_threshold_by_regime", "Cost-optimal threshold by regime and model")

_main = primary[primary.model != "baseline"][
    ["model", "strategy", "threshold", "target_met", "auc_pr", "auc_pr_lift", "roc_auc", "precision", "recall", "f1",
     "accuracy", "alert_rate", "value_detection_rate", "cost_per_txn_gbp", "saving_vs_default_pct"]
].copy()
_main["model"] = _main["model"].map(MODEL_LABELS).fillna(_main["model"])
export_table(_main, "table1_main_results", "Threshold selection strategies under the UK regulatory cost regime, IEEE-CIS.")

_reg = regime_table[regime_table.model != "baseline"][
    ["regime", "model", "threshold", "precision", "recall", "alert_rate", "cost_per_txn_gbp", "saving_vs_default_pct"]
].copy()
_reg["model"] = _reg["model"].map(MODEL_LABELS).fillna(_reg["model"])
export_table(_reg, "table2_regime_comparison", "Cost-optimal threshold by regulatory cost regime, identical data and models.")

_rob_parts = [primary.assign(scenario="IEEE, stratified"), temporal.assign(scenario="IEEE, temporal"), ulb.assign(scenario="ULB, stratified")]
if paysim is not None:
    _rob_parts.append(paysim.assign(scenario="PaySim, stratified"))
_rob = pd.concat(_rob_parts, ignore_index=True)
_rob = _rob[(_rob.strategy == "cost_optimal") & (_rob.model != "baseline")][
    ["scenario", "model", "threshold", "auc_pr", "auc_pr_lift", "precision", "recall", "cost_per_txn_gbp", "saving_vs_default_pct"]
].copy()
_rob["model"] = _rob["model"].map(MODEL_LABELS).fillna(_rob["model"])
export_table(_rob, "table3_robustness", "Robustness of the cost-optimal threshold to split rule and dataset.")

export_table(sweep_fp, "table4_sensitivity_fp_cost", "Sensitivity of the cost-optimal threshold to the false-positive cost proxy.")
export_table(sweep_cap, "table5_sensitivity_cap", "Sensitivity of the cost-optimal threshold to the PSR reimbursement cap.")
export_table(sweep_app, "table6_sensitivity_app_uplift", "Cost-optimal threshold under APP loss-magnitude scenarios.")
export_table(seed_table, "table7_seed_stability", "Stability of the cost-optimal threshold across random seeds.")
export_table(sample_size_table, "table8_sample_size_ablation", "Effect of subsample size on AUC-PR and the cost-optimal threshold, XGBoost.")
export_table(sweep_liability, "table9_sensitivity_liability_share", "Sensitivity of the cost-optimal threshold to the PSR sending-PSP liability share.")
export_table(sweep_admin, "table10_sensitivity_claim_admin", "Sensitivity of the cost-optimal threshold to the claim-handling admin cost.")
export_table(significance_table, "table11_bootstrap_significance", "Bootstrap significance test: literature vs uk_regulatory cost-optimal threshold.")
if not shap_importance.empty:
    export_table(shap_importance.head(25), "table12_shap_top_features", "Top 25 features by mean absolute SHAP value, primary XGBoost model.")
export_table(cost_ci_table, "table13_cost_confidence_intervals", "Bootstrap 95% CI on realised test-set cost per transaction, XGBoost.")

for _key, _trace in TUNING_TRACES.items():
    if "tuning_search" in _key:
        _stem = "table14_hp_search_" + _key.split("|")[1]
        export_table(_trace, _stem, f"Hyperparameter search trace (150k-row subsample): {_key}", float_fmt="%.5f")

export_table(imbalance_table, "table15_imbalance_method_comparison",
             "Imbalance-handling method comparison, XGBoost, tuning subsample, default hyperparameters.")
export_table(stack_table, "table16_stacked_ensemble",
             "Logistic-stacked ensemble of the three proposal models vs. the best single model, IEEE-CIS test set.")

all_results().to_csv(OUT_DIR / "all_results_raw.csv", index=False)
print(f"\nraw results -> {OUT_DIR / 'all_results_raw.csv'} ({len(all_results())} rows)")

figure -> /content/outputs/figures/fig1_precision_recall.png
figure -> /content/outputs/figures/fig2_roc.png
figure -> /content/outputs/figures/fig3_calibration.png
figure -> /content/outputs/figures/fig4_cost_by_regime.png
figure -> /content/outputs/figures/fig5_sensitivity.png
figure -> /content/outputs/figures/fig6_shap_importance.png
figure -> /content/outputs/figures/fig7_threshold_by_regime.png
table -> table1_main_results.csv, table1_main_results.tex
table -> table2_regime_comparison.csv, table2_regime_comparison.tex
table -> table3_robustness.csv, table3_robustness.tex
table -> table4_sensitivity_fp_cost.csv, table4_sensitivity_fp_cost.tex
table -> table5_sensitivity_cap.csv, table5_sensitivity_cap.tex
table -> table6_sensitivity_app_uplift.csv, table6_sensitivity_app_uplift.tex
table -> table7_seed_stability.csv, table7_seed_stability.tex
table -> table8_sample_size_ablation.csv, table8_sample_size_ablation.tex
table -> table9_sensitivity_liability_share.csv, table9_sensitivit

In [ ]:
# =============================================================================
# CELL 30 - EXPORT AND BACKUP
# =============================================================================

_stamp = time.strftime("%Y%m%d_%H%M")
_zip_dir = Path("/content") if IN_COLAB else OUT_DIR.parent
_zip = _zip_dir / f"fraud_thresholds_{STUDENT_ID}_v7_{_stamp}.zip"

with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for _f in OUT_DIR.rglob("*"):
        if _f.is_file():
            zf.write(_f, _f.relative_to(OUT_DIR))
print(f"{_zip} ({_zip.stat().st_size / 1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(_zip))

(OUT_DIR / "run_config.json").write_text(json.dumps({
    "student_id": STUDENT_ID, "timestamp": _stamp, "version": "v7",
    "imbalance_method_comparison": imbalance_table.to_dict(orient="records"),
    "best_imbalance_method": BEST_IMBALANCE_METHOD,
    "stack_ensemble": stack_table.to_dict(orient="records")[0],
    "tuning_sample_rows": TUNING_SAMPLE_ROWS, "tuned_cfg": asdict(tuned_cfg),
    "primary": asdict(primary_cfg), "regimes": {k: {kk: vv for kk, vv in v.items()} for k, v in REGIMES.items()},
    "psr_cap_gbp": PSR_CAP_GBP, "psr_liability_share": PSR_LIABILITY_SHARE,
    "fp_cost_sweep": FP_COST_SWEEP, "cap_sweep": CAP_SWEEP, "app_uplift_sweep": APP_UPLIFT_SWEEP,
    "liability_share_sweep": LIABILITY_SHARE_SWEEP, "claim_admin_sweep": CLAIM_ADMIN_SWEEP,
    "sample_size_ablation": [s or "full" for s in SAMPLE_SIZE_ABLATION],
}, indent=2, default=str))
print("config recorded")

/content/fraud_thresholds_5753132_v7_20260823_2011.zip (1.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

config recorded
